# Cross-backbone comparison, ablations, diagnostics

Consumes the cached LLM probabilities written by `02_pipeline_*.ipynb`.
Produces Tables 4-7, the K x lambda ablation, pairwise McNemar tests, the
three diagnostics of section 7.5, and the Aurora co-occurrence matrix.

CPU only; no GPU runtime needed.


In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
# All paths derive from one root: $SDG_ROOT, else config.yaml, else Drive on
# Colab, else ./sdg_data. See README section 1.
import os, sys

def _resolve_sdg_root():
    r = os.environ.get('SDG_ROOT')
    if r:
        return os.path.abspath(os.path.expanduser(r))
    for cand in ('config.yaml', os.path.join('..', 'config.yaml')):
        if os.path.exists(cand):
            with open(cand) as fh:
                for line in fh:
                    line = line.split('#')[0].strip()
                    if line.startswith('root:'):
                        v = line.split(':', 1)[1].strip().strip('"\'')
                        if v:
                            return os.path.abspath(os.path.expanduser(v))
    if 'google.colab' in sys.modules or os.path.isdir('/content'):
        try:
            from google.colab import drive
            if not os.path.exists('/content/drive/MyDrive'):
                drive.mount('/content/drive')
            return '/content/drive/MyDrive/sdg-llm-graph'
        except Exception:
            pass
    return os.path.abspath('./sdg_data')

SDG_ROOT   = _resolve_sdg_root()
GRAPH_DIR  = os.path.join(SDG_ROOT, 'sdggraph')
DRIVE_ROOT = os.path.join(SDG_ROOT, 'aurora_sdg_graph_full')
DATA_CACHE = os.path.join(DRIVE_ROOT, 'data_cache')
for _d in (SDG_ROOT, GRAPH_DIR, DRIVE_ROOT, DATA_CACHE):
    os.makedirs(_d, exist_ok=True)

print(f'SDG_ROOT   : {SDG_ROOT}')
print(f'GRAPH_DIR  : {GRAPH_DIR}')
print(f'DATA_CACHE : {DATA_CACHE}')

# Sanity peek: do all four backbones' cached goal probabilities look sane?
import numpy as np
ROOT = DRIVE_ROOT
for m in ['gemma4-26b','mistral-24b','qwen3-32b','mixtral-8x7b']:
    p = os.path.join(ROOT, f'results_v5_multi_{m}_n10000',
                     f'goal_probs_test_v5_multi_{m}_n10000.npy')
    if not os.path.exists(p):
        print(f'{m:13s} MISSING: {p}')
        continue
    g = np.load(p)                                   # (10000, 17)
    print(f'{m:13s} mean={g.mean():.3f}  exact0={(g==0).mean():.2%}  '
          f'rows>=14 zeros={((g==0).sum(1)>=14).mean():.2%}  '
          f'nonzero/paper={(g>0).sum(1).mean():.1f}')


In [ ]:
for m in ['gemma4-26b','mistral-24b']:
    g = np.load(f'{ROOT}/results_v5_multi_{m}_n10000/goal_probs_test_v5_multi_{m}_n10000.npy')
    print(f'{m:13s} per-paper max: mean={g.max(1).mean():.3f}  '
          f'>0.5={ (g.max(1)>0.5).mean():.1%}  >0.7={(g.max(1)>0.7).mean():.1%}')

In [ ]:
!pip -q install sentence-transformers scikit-learn pandas numpy

import os, json, re, time, hashlib
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from sentence_transformers import SentenceTransformer
import warnings; warnings.filterwarnings('ignore')

SEED        = 42
N_TEST      = 10000
N_VAL       = 200
N_FEWSHOT   = 40
np.random.seed(SEED)

MODELS = ['gemma4-26b', 'mistral-24b', 'qwen3-32b', 'mixtral-8x7b']
print(f'Models in scope: {MODELS}')


## 1. Load splits — read abstracts_full.parquet directly, slice sequentially

In [ ]:

ABSTRACTS_CACHE = os.path.join(DATA_CACHE, 'abstracts_full.parquet')
assert os.path.exists(ABSTRACTS_CACHE), \
    f'Missing {ABSTRACTS_CACHE} — run any of the per-model notebooks first.'

df_papers = pd.read_parquet(ABSTRACTS_CACHE)
print(f'Loaded abstracts_full.parquet: {len(df_papers)} papers')

# Defensive: tolerate a smaller-than-expected parquet (some pre-v3 runs had
# this happen). We just take what's there and adjust N_TEST accordingly.
expected = N_FEWSHOT + N_VAL + N_TEST
if len(df_papers) < expected:
    print(f'  WARNING: parquet has {len(df_papers)} rows, expected {expected}.')
    print(f'  Adjusting N_TEST: {N_TEST} → {len(df_papers) - N_FEWSHOT - N_VAL}')
    N_TEST = len(df_papers) - N_FEWSHOT - N_VAL

# Sequential slice — matches the main notebook exactly
df_fewshot = df_papers.iloc[:N_FEWSHOT].reset_index(drop=True)
df_val     = df_papers.iloc[N_FEWSHOT : N_FEWSHOT+N_VAL].reset_index(drop=True)
df_test    = df_papers.iloc[N_FEWSHOT+N_VAL : N_FEWSHOT+N_VAL+N_TEST].reset_index(drop=True)
print(f'Splits: fewshot={len(df_fewshot)}, val={len(df_val)}, test={len(df_test)}')

# Goal columns (must come from the parquet, not from a re-derivation)
goal_cols = sorted([c for c in df_papers.columns if re.match(r'^sdg_\d+$', c)],
                   key=lambda c: int(c.split('_')[1]))
assert len(goal_cols) == 17, f'Expected 17 goal columns, got {len(goal_cols)}'
labels_test = df_test[goal_cols].values.astype(np.int8)
labels_val  = df_val[goal_cols].values.astype(np.int8)

def _hash_dois(df):
    dois = df['doi'].astype(str).tolist()
    return hashlib.sha1(''.join(dois).encode('utf-8')).hexdigest()[:12]

test_hash = _hash_dois(df_test)
val_hash  = _hash_dois(df_val)
print(f'\n── Data fingerprint ──')
print(f'  test split: hash={test_hash}  ({len(df_test)} rows)')
print(f'  val  split: hash={val_hash}  ({len(df_val)} rows)')

ORIGINAL_HASHES = {'test': '4c5581303537', 'val': 'b212c22b4951'}
match_test = test_hash == ORIGINAL_HASHES['test']
match_val  = val_hash  == ORIGINAL_HASHES['val']

if match_test and match_val:
    print('  ✓ matches original runs — absolute F1 numbers will reproduce')
else:
    print('  ⚠ does NOT match the original-run fingerprint:')
    print(f'    expected test={ORIGINAL_HASHES["test"]}, val={ORIGINAL_HASHES["val"]}')
    print(f'    got      test={test_hash},     val={val_hash}')
    print('  This is OK — it just means abstracts_full.parquet was rebuilt')
    print('  since the per-model runs. Cached goal_probs_*.npy are still')
    print('  aligned with the current parquet (Stage 1+2 reads the same parquet')
    print('  in sequence). Cross-model comparisons remain valid; absolute F1')
    print('  may differ from the per-run notebooks because the test set differs.')


## 2. Load all 4 models' cached probs + entity cache

In [ ]:
cached = {}
mismatches = []
for m in MODELS:
    rd = os.path.join(DRIVE_ROOT, f'results_v5_multi_{m}_n10000')
    g_test_path = os.path.join(rd, f'goal_probs_test_v5_multi_{m}_n10000.npy')
    g_val_path  = os.path.join(rd, f'goal_probs_val_v5_multi_{m}_n10000.npy')
    t_test_path = os.path.join(rd, f'target_probs_test_v5_multi_{m}_n10000.npy')
    t_val_path  = os.path.join(rd, f'target_probs_val_v5_multi_{m}_n10000.npy')
    for p in [g_test_path, g_val_path, t_test_path, t_val_path]:
        assert os.path.exists(p), f'Missing {p}'
    g_test = np.load(g_test_path)
    g_val  = np.load(g_val_path)
    t_test = np.load(t_test_path)
    t_val  = np.load(t_val_path)
    cached[m] = {'goal_test':g_test, 'goal_val':g_val,
                 'target_test':t_test, 'target_val':t_val}
    # Sanity vs current df sizes
    if g_test.shape[0] != len(df_test):
        mismatches.append((m, 'goal_test', g_test.shape[0], len(df_test)))
    if g_val.shape[0]  != len(df_val):
        mismatches.append((m, 'goal_val',  g_val.shape[0],  len(df_val)))
    print(f'{m:15s}  goal_test={g_test.shape}  target_test={t_test.shape}')

if mismatches:
    print('\n⚠ Shape mismatches between cached probs and current splits:')
    for m, what, got, expected in mismatches:
        print(f'  {m} {what}: cached has {got} rows, current df has {expected}')
    print('  This means the abstracts parquet was resized after the run.')
    print('  Cannot proceed — re-run those models with the current parquet.')
    raise RuntimeError('Cached probs do not match current splits.')

# Hash-keyed entity cache (shared across models)
ent_test_path = os.path.join(DATA_CACHE, f'entities_test_hash_{test_hash}.json')
ent_val_path  = os.path.join(DATA_CACHE, f'entities_val_hash_{val_hash}.json')
if os.path.exists(ent_test_path) and os.path.exists(ent_val_path):
    with open(ent_test_path) as f: entities_test = json.load(f)
    with open(ent_val_path)  as f: entities_val  = json.load(f)
    print(f'\nEntity cache (hash-keyed, current splits): '
          f'{len(entities_test)} test, {len(entities_val)} val')
else:
    # Fall back to the original-run hash if available — entities aren't strictly
    # needed for the K/lambda ablation (they're for doc_entities only).
    fallback_test = os.path.join(DATA_CACHE, f'entities_test_hash_{ORIGINAL_HASHES["test"]}.json')
    fallback_val  = os.path.join(DATA_CACHE, f'entities_val_hash_{ORIGINAL_HASHES["val"]}.json')
    if os.path.exists(fallback_test) and os.path.exists(fallback_val):
        print(f'\nEntity cache for current hash not found — using original-run hash files.')
        print(f'  WARNING: indices may not align if df_test changed. doc_entities will be skipped.')
        entities_test, entities_val = None, None
    else:
        print(f'\nNo entity cache available. doc_entities will be skipped.')
        entities_test, entities_val = None, None


## 3. Target hierarchy + SBERT

In [ ]:
# Target hierarchy (no Pradhan needed for the ablation — we only do graphs in headline)
TARGETS_PER_GOAL = [7,8,13,10,9,8,5,12,8,10,10,11,5,10,12,12,19]
TARGET_IDS = []
GOAL_TO_TARGETS = {g: [] for g in range(1,18)}
for g_idx, n in enumerate(TARGETS_PER_GOAL):
    g = g_idx + 1
    for t in range(1, n+1):
        GOAL_TO_TARGETS[g].append(len(TARGET_IDS))
        TARGET_IDS.append(f'{g}.{t}')

def aggregate_targets_to_goals(target_probs):
    out = np.zeros((target_probs.shape[0], 17), dtype=np.float32)
    for g in range(1, 18):
        idxs = GOAL_TO_TARGETS[g]
        out[:, g-1] = target_probs[:, idxs].max(axis=1)
    return out

print('Target hierarchy ready.')
sbert = SentenceTransformer('all-MiniLM-L6-v2')
print('SBERT loaded.')


## 4. Neighbour pool + encode val/test

In [ ]:
OPENALEX_MAILTO = os.environ.get('OPENALEX_MAILTO', 'sdg-research@example.com')
# Neighbour pool — read from cache, or rebuild if missing.
N_NEIGHBOUR_POOL = 5000
pool_cache = os.path.join(DATA_CACHE,
    f'neighbour_pool_abstracts_n{N_NEIGHBOUR_POOL}_seed{SEED}.parquet')

if os.path.exists(pool_cache):
    df_pool = pd.read_parquet(pool_cache)
    df_pool = df_pool[df_pool['abstract'].fillna('').str.len() > 50].reset_index(drop=True)
    print(f'Neighbour pool loaded from cache: {len(df_pool)} docs')
else:
    # Rebuild from aurora_multilabel_full.parquet — pool is disjoint from the
    # train/val/test splits and from a multi-label subset to keep label density.
    print(f'Pool cache not found at {pool_cache}')
    print(f'Rebuilding from aurora_multilabel_full.parquet …')
    import requests, time

    aurora_path = os.path.join(DATA_CACHE, 'aurora_multilabel_full.parquet')
    assert os.path.exists(aurora_path), f'Missing {aurora_path}'
    df_multi = pd.read_parquet(aurora_path)
    df_multi['_doi_lower'] = df_multi['doi'].astype(str).str.strip().str.lower()
    print(f'  Aurora multi-label: {len(df_multi)} papers')

    # Disjoint from current splits — drop any DOI in fewshot/val/test
    used_dois = set(
        list(df_fewshot['doi'].astype(str).str.strip().str.lower()) +
        list(df_val['doi'].astype(str).str.strip().str.lower()) +
        list(df_test['doi'].astype(str).str.strip().str.lower())
    )
    df_pool_candidates = df_multi[~df_multi['_doi_lower'].isin(used_dois)]
    print(f'  Candidates after dropping current splits: {len(df_pool_candidates)}')

    # Sample with the same SEED the per-run notebooks used
    if len(df_pool_candidates) >= N_NEIGHBOUR_POOL:
        df_pool = df_pool_candidates.sample(N_NEIGHBOUR_POOL, random_state=SEED).reset_index(drop=True)
    else:
        df_pool = df_pool_candidates.reset_index(drop=True)
    print(f'  Pool sample: {len(df_pool)} DOIs — fetching abstracts from OpenAlex…')

    # OpenAlex fetcher (lifted from the per-run notebook)
    def fetch_abstracts_openalex(dois, batch_size=50):
        results = {}
        dois = [d.strip().lower() for d in dois if d and str(d) != 'nan']
        for i in range(0, len(dois), batch_size):
            batch = dois[i:i+batch_size]
            url = (f'https://api.openalex.org/works?filter=doi:{"|".join(batch)}'
                   f'&select=doi,title,abstract_inverted_index'
                   f'&per-page={batch_size}&mailto={OPENALEX_MAILTO}')
            try:
                r = requests.get(url, timeout=30, headers={'User-Agent': 'sdg-research/1.0'})
                if r.status_code == 200:
                    for w in r.json().get('results', []):
                        doi = w.get('doi','').replace('https://doi.org/','').lower()
                        title = w.get('title') or ''
                        inv = w.get('abstract_inverted_index') or {}
                        if inv:
                            idx2word = {p: word for word, ps in inv.items() for p in ps}
                            abstract = ' '.join(idx2word[k] for k in sorted(idx2word))
                        else:
                            abstract = ''
                        if title:
                            results[doi] = {'title': title, 'abstract': abstract}
            except Exception as e:
                print(f'  Batch {i//batch_size} error: {e}')
            time.sleep(0.1)
            if (i // batch_size) % 10 == 0:
                print(f'  Fetched {min(i+batch_size, len(dois))}/{len(dois)}')
        return results

    abstract_dict = fetch_abstracts_openalex(df_pool['doi'].astype(str).tolist())
    df_pool['title']    = df_pool['_doi_lower'].map(lambda d: abstract_dict.get(d, {}).get('title',''))
    df_pool['abstract'] = df_pool['_doi_lower'].map(lambda d: abstract_dict.get(d, {}).get('abstract',''))
    df_pool = df_pool.drop(columns=['_doi_lower'])
    df_pool = df_pool[df_pool['abstract'].fillna('').str.len() > 50].reset_index(drop=True)
    df_pool.to_parquet(pool_cache, index=False)
    print(f'  Saved {len(df_pool)} pool papers with abstracts to {pool_cache}')

# ── Continues exactly as before ──
pool_labels_goal = df_pool[goal_cols].values.astype(np.float32)
print(f'Pool label density: {pool_labels_goal.mean():.3f}')

print('Encoding pool with SBERT (~30s for 3-5k docs)...')
pool_text = (df_pool['title'].fillna('') + '. ' + df_pool['abstract'].fillna('')).tolist()
pool_emb  = sbert.encode(pool_text, normalize_embeddings=True, batch_size=64,
                          show_progress_bar=False).astype(np.float32)
print('Encoding val + test...')
val_text  = (df_val['title'].fillna('')  + '. ' + df_val['abstract'].fillna('')).tolist()
test_text = (df_test['title'].fillna('') + '. ' + df_test['abstract'].fillna('')).tolist()
val_emb   = sbert.encode(val_text,  normalize_embeddings=True, batch_size=64,
                          show_progress_bar=False).astype(np.float32)
test_emb  = sbert.encode(test_text, normalize_embeddings=True, batch_size=64,
                          show_progress_bar=False).astype(np.float32)
print(f'Embeddings: pool {pool_emb.shape}, val {val_emb.shape}, test {test_emb.shape}')

## 5. Helper functions

In [ ]:
def opt_threshold_per_class(probs, labels):
    grid = np.arange(0.05, 0.96, 0.05)
    C = probs.shape[1]
    thr = np.zeros(C, dtype=np.float32)
    for c in range(C):
        best_f1, best_t = -1, 0.5
        for t in grid:
            preds = (probs[:, c] >= t).astype(int)
            f = f1_score(labels[:, c], preds, zero_division=0)
            if f > best_f1: best_f1, best_t = f, t
        thr[c] = best_t
    return thr

def macro_f1(probs, labels, thr):
    preds = (probs >= thr).astype(int)
    return f1_score(labels, preds, average='macro', zero_division=0)

def doc_neighbours_evidence(query_emb, pool_emb, pool_labels, K):
    sims = query_emb @ pool_emb.T
    nn_idx = np.argsort(-sims, axis=1)[:, :K]
    out = np.zeros((query_emb.shape[0], 17), dtype=np.float32)
    for i in range(query_emb.shape[0]):
        out[i] = pool_labels[nn_idx[i]].mean(axis=0)
    return out

def calibrate_lambda_global(prior, evidence, labels):
    best_lam, best_f1 = 0.0, -1.0
    for lam in np.arange(0.0, 1.01, 0.1):
        combo = (1 - lam) * prior + lam * evidence
        thr = opt_threshold_per_class(combo, labels)
        f = macro_f1(combo, labels, thr)
        if f > best_f1: best_f1, best_lam = f, lam
    return best_lam, best_f1

def calibrate_per_goal_lambda_cv(prior, evidence, labels, K_folds=5, seed=SEED):
    from sklearn.model_selection import KFold
    n, C = prior.shape
    lams = np.zeros(C, dtype=np.float32)
    grid = np.arange(0.0, 1.01, 0.05)
    thr_grid = np.arange(0.05, 0.96, 0.05)
    for c in range(C):
        kf = KFold(n_splits=K_folds, shuffle=True, random_state=seed)
        fold_best = []
        for tr, te in kf.split(np.arange(n)):
            best_lam, best_f1 = 0.0, -1
            for lam in grid:
                combo = (1 - lam) * prior[te, c] + lam * evidence[te, c]
                f_t = max(f1_score(labels[te, c], (combo >= t).astype(int), zero_division=0)
                          for t in thr_grid)
                if f_t > best_f1: best_f1, best_lam = f_t, lam
            fold_best.append(best_lam)
        lams[c] = np.median(fold_best)
    return lams


## 6. Cross-model headline table

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# CROSS-MODEL HEADLINE TABLE
# ────────────────────────────────────────────────────────────────────────────
headline = []
for m in MODELS:
    g_t = cached[m]['goal_test']; g_v = cached[m]['goal_val']
    t_t = cached[m]['target_test']; t_v = cached[m]['target_val']

    thr = opt_threshold_per_class(g_v, labels_val)
    no_graph = macro_f1(g_t, labels_test, thr)

    g_t_agg = aggregate_targets_to_goals(t_t); g_v_agg = aggregate_targets_to_goals(t_v)
    thr_a = opt_threshold_per_class(g_v_agg, labels_val)
    tgt_agg = macro_f1(g_t_agg, labels_test, thr_a)

    K_HEADLINE = 50
    evid_v = doc_neighbours_evidence(val_emb,  pool_emb, pool_labels_goal, K=K_HEADLINE)
    evid_t = doc_neighbours_evidence(test_emb, pool_emb, pool_labels_goal, K=K_HEADLINE)
    lam_g, _ = calibrate_lambda_global(g_v, evid_v, labels_val.astype(np.float32))
    combo_v = (1 - lam_g) * g_v + lam_g * evid_v
    combo_t = (1 - lam_g) * g_t + lam_g * evid_t
    thr_d = opt_threshold_per_class(combo_v, labels_val)
    dn = macro_f1(combo_t, labels_test, thr_d)

    headline.append({'model': m, 'no_graph': no_graph, 'target_agg': tgt_agg,
                      'doc_neighbours': dn})

df_head = pd.DataFrame(headline).set_index('model')
print('\n── HEADLINE COMPARISON (test macro-F1) ──')
print(df_head.round(4).to_string())

print('\nBest per metric:')
for metric, model in df_head.idxmax().items():
    print(f'  {metric:18s} → {model:15s}  (F1={df_head.loc[model, metric]:.4f})')


## 6b. Cross-backbone graph propagation (Tables 3 + 4)


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# v5 NEW: CROSS-MODEL GRAPH PROPAGATION TABLE (Tables 3 + 4 in the paper)
# ────────────────────────────────────────────────────────────────────────────
# Aggregates each backbone's graph_comparison_v5_multi_<m>_n10000.csv (written
# by the multimodel notebook's Cell 27) into a single backbone x matrix view.
# Pre-v5 this aggregation was done by hand from the per-model CSVs; v5 wires
# it into the ablate notebook so Tables 3 (goal-level) and 4 (target-level)
# are reproducible from a single run of this notebook.

import os, pandas as pd

graph_rows = []
for m in MODELS:
    rd = os.path.join(DRIVE_ROOT, f'results_v5_multi_{m}_n10000')
    csv_path = os.path.join(rd, f'graph_comparison_v5_multi_{m}_n10000.csv')
    if not os.path.exists(csv_path):
        # Tolerate missing CSVs (the headline / ablation table is still
        # produced for other models). Print a clear warning instead of
        # raising — the per-model graph step is independent.
        print(f'  WARNING: {csv_path} not found — graph results for {m} will be missing.')
        continue
    df_g = pd.read_csv(csv_path)
    # Each row is one (graph, alpha, conf, [w_B]) configuration with val/test F1.
    # We take the val-best per graph, then record its test F1 (the paper's
    # convention).
    val_best = df_g.sort_values('val_f1', ascending=False).groupby('graph').head(1)
    for _, row in val_best.iterrows():
        graph_rows.append({
            'model':         m,
            'graph':         row['graph'],
            'alpha':         row['alpha'],
            'conf':          row['conf'],
            'w_B':           row.get('w_B'),
            'val_f1':        row['val_f1'],
            'test_f1':       row['test_f1'],
            'val_f1_target': row.get('val_f1_target'),
            'test_f1_target': row.get('test_f1_target'),
        })

df_graphs = pd.DataFrame(graph_rows)
if df_graphs.empty:
    print('No per-model graph CSVs found yet — run the multimodel notebook first.')
else:
    print('\n── CROSS-BACKBONE GRAPH PROPAGATION (val-best per matrix per model) ──')
    pivot_goal = df_graphs.pivot_table(index='graph', columns='model',
                                         values='test_f1').round(4)
    print('Goal-level test macro-F1 (Table 3 columns A-G x 4 backbones):')
    print(pivot_goal.to_string())

    if df_graphs['test_f1_target'].notna().any():
        pivot_tgt = df_graphs.pivot_table(index='graph', columns='model',
                                            values='test_f1_target').round(4)
        print('\nTarget-level test macro-F1 (Table 4 — target-level matrices only):')
        print(pivot_tgt.dropna(how='all').to_string())

    # Save the cross-backbone graph table for the paper
    out_dir = os.path.join(DRIVE_ROOT, 'cross_model_comparison_v5')
    os.makedirs(out_dir, exist_ok=True)
    df_graphs.to_csv(os.path.join(out_dir, 'cross_backbone_graphs.csv'), index=False)
    pivot_goal.to_csv(os.path.join(out_dir, 'cross_backbone_graphs_pivot_goal.csv'))
    if df_graphs['test_f1_target'].notna().any():
        pivot_tgt.to_csv(os.path.join(out_dir,
                                       'cross_backbone_graphs_pivot_target.csv'))
    print(f'\nSaved cross-backbone graph tables to {out_dir}/')


## 7. Ablation: K × per-goal λ

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# ABLATION: doc_neighbours K x PER_GOAL_LAMBDA
# ────────────────────────────────────────────────────────────────────────────
K_GRID = [5, 10, 20, 50]
ablation_rows = []

# Pre-compute evidence per K (shared across models — depends only on SBERT)
evid_per_k_test = {K: doc_neighbours_evidence(test_emb, pool_emb, pool_labels_goal, K=K) for K in K_GRID}
evid_per_k_val  = {K: doc_neighbours_evidence(val_emb,  pool_emb, pool_labels_goal, K=K) for K in K_GRID}

for m in MODELS:
    g_t = cached[m]['goal_test']; g_v = cached[m]['goal_val']

    for K in K_GRID:
        evid_v, evid_t = evid_per_k_val[K], evid_per_k_test[K]

        # Variant A: global lambda
        lam_g, f1_v_g = calibrate_lambda_global(g_v, evid_v, labels_val.astype(np.float32))
        combo_v_g = (1 - lam_g) * g_v + lam_g * evid_v
        combo_t_g = (1 - lam_g) * g_t + lam_g * evid_t
        thr_g = opt_threshold_per_class(combo_v_g, labels_val)
        f1_t_g = macro_f1(combo_t_g, labels_test, thr_g)

        # Variant B: per-goal lambda
        lams = calibrate_per_goal_lambda_cv(g_v, evid_v, labels_val.astype(np.float32))
        combo_v_pg = (1 - lams)[None,:] * g_v + lams[None,:] * evid_v
        combo_t_pg = (1 - lams)[None,:] * g_t + lams[None,:] * evid_t
        thr_pg = opt_threshold_per_class(combo_v_pg, labels_val)
        f1_v_pg = macro_f1(combo_v_pg, labels_val, thr_pg)
        f1_t_pg = macro_f1(combo_t_pg, labels_test, thr_pg)

        ablation_rows.append({'model': m, 'K': K, 'lambda': 'global',
                              'val_f1': f1_v_g,  'test_f1': f1_t_g,  'mean_lambda': lam_g})
        ablation_rows.append({'model': m, 'K': K, 'lambda': 'per_goal',
                              'val_f1': f1_v_pg, 'test_f1': f1_t_pg, 'mean_lambda': float(lams.mean())})

df_abl = pd.DataFrame(ablation_rows)
print('\n── ABLATION — test_f1 by model × K × lambda ──')
print(df_abl.pivot_table(index=['model','K'], columns='lambda', values='test_f1').round(4).to_string())

print('\n── BEST K per model (per_goal λ branch) ──')
best_K = (df_abl[df_abl['lambda']=='per_goal']
          .sort_values('test_f1', ascending=False)
          .groupby('model').first())[['K','test_f1','mean_lambda']]
print(best_K.to_string())

# Save outputs to a versioned dir
out_dir = os.path.join(DRIVE_ROOT, 'cross_model_comparison_v5')
os.makedirs(out_dir, exist_ok=True)
df_head.to_csv(os.path.join(out_dir, 'headline.csv'))
df_abl.to_csv(os.path.join(out_dir, 'ablation_K_lambda.csv'), index=False)
print(f'\nSaved CSVs to {out_dir}/')


## 8. Final summary

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# FINAL SUMMARY
# ────────────────────────────────────────────────────────────────────────────
print('\n' + '='*72)
print('FINAL — best macro-F1 per model (across all methods + ablations)')
print('='*72)

best_overall = {}
for m in MODELS:
    candidates = [
        ('no_graph',                   df_head.loc[m, 'no_graph']),
        ('target_aggregated',          df_head.loc[m, 'target_agg']),
        ('doc_neighbours (canonical)', df_head.loc[m, 'doc_neighbours']),
    ]
    sub = df_abl[df_abl['model'] == m]
    if len(sub):
        best_row = sub.loc[sub['test_f1'].idxmax()]
        candidates.append((f'doc_neighbours (K={int(best_row["K"])}, λ={best_row["lambda"]})',
                           best_row['test_f1']))
    name, f1 = max(candidates, key=lambda x: x[1])
    best_overall[m] = (name, f1)
    print(f'\n  {m:15s}  best: {name:50s}  F1={f1:.4f}')

winner = max(best_overall.items(), key=lambda kv: kv[1][1])
print(f'\n{"="*72}')
print(f'WINNER: {winner[0]} with F1={winner[1][1]:.4f} via {winner[1][0]}')
print(f'{"="*72}')


In [ ]:
# 169 SDG target descriptors (paraphrased from UN official text).
# Used by doc_entities for entity↔target keyword overlap.
TARGET_DESC = {
    # SDG 1 — No Poverty (7 targets)
    '1.1': 'eradicate extreme poverty for all people everywhere measured as living on less than 1.25 dollars a day',
    '1.2': 'reduce by half the proportion of men women and children of all ages living in poverty',
    '1.3': 'implement nationally appropriate social protection systems and measures for all',
    '1.4': 'ensure all men and women have equal rights to economic resources access to basic services ownership and control over land',
    '1.5': 'build resilience of the poor and vulnerable reduce exposure to climate-related extreme events economic social environmental shocks disasters',
    '1.a': 'mobilization of resources from a variety of sources including enhanced development cooperation to provide adequate predictable means for developing countries',
    '1.b': 'create sound policy frameworks at national regional international levels based on pro-poor and gender-sensitive development strategies',
    # SDG 2 — Zero Hunger (8 targets)
    '2.1': 'end hunger ensure access by all people to safe nutritious sufficient food all year round',
    '2.2': 'end all forms of malnutrition stunting wasting children under 5 nutritional needs adolescent girls pregnant lactating women older persons',
    '2.3': 'double agricultural productivity incomes of small-scale food producers women indigenous family farmers pastoralists fishers',
    '2.4': 'ensure sustainable food production systems implement resilient agricultural practices increase productivity',
    '2.5': 'maintain genetic diversity of seeds cultivated plants farmed domesticated animals related wild species',
    '2.a': 'increase investment rural infrastructure agricultural research extension services technology development plant livestock gene banks',
    '2.b': 'correct prevent trade restrictions distortions world agricultural markets parallel elimination all forms agricultural export subsidies',
    '2.c': 'adopt measures ensure proper functioning food commodity markets derivatives facilitate timely access market information',
    # SDG 3 — Good Health and Well-being (13 targets)
    '3.1': 'reduce global maternal mortality ratio to less than 70 per 100000 live births',
    '3.2': 'end preventable deaths newborns children under 5 years neonatal under-5 mortality',
    '3.3': 'end epidemics AIDS tuberculosis malaria neglected tropical diseases combat hepatitis water-borne diseases other communicable',
    '3.4': 'reduce by one third premature mortality from non-communicable diseases prevention treatment promote mental health well-being',
    '3.5': 'strengthen prevention treatment of substance abuse narcotic drug abuse harmful use of alcohol',
    '3.6': 'halve number global deaths injuries from road traffic accidents',
    '3.7': 'ensure universal access to sexual and reproductive health-care services family planning information education',
    '3.8': 'achieve universal health coverage financial risk protection access to quality essential health-care services medicines vaccines',
    '3.9': 'reduce deaths illnesses from hazardous chemicals air water soil pollution contamination',
    '3.a': 'strengthen implementation of WHO Framework Convention on Tobacco Control all countries',
    '3.b': 'support research development of vaccines medicines for communicable non-communicable diseases affecting developing countries',
    '3.c': 'increase health financing recruitment development training retention health workforce developing countries',
    '3.d': 'strengthen capacity for early warning risk reduction management of national global health risks',
    # SDG 4 — Quality Education (10 targets)
    '4.1': 'ensure all girls boys complete free equitable quality primary secondary education leading to relevant effective learning outcomes',
    '4.2': 'ensure all girls boys access quality early childhood development care pre-primary education ready primary',
    '4.3': 'ensure equal access for all women men affordable quality technical vocational tertiary education university',
    '4.4': 'increase number of youth adults relevant skills technical vocational employment decent jobs entrepreneurship',
    '4.5': 'eliminate gender disparities education ensure equal access all levels education vocational training vulnerable persons disabilities indigenous',
    '4.6': 'ensure all youth substantial proportion of adults achieve literacy numeracy',
    '4.7': 'ensure all learners acquire knowledge skills needed to promote sustainable development education for sustainable lifestyles human rights gender equality',
    '4.a': 'build upgrade education facilities child disability gender sensitive provide safe non-violent inclusive effective learning environments',
    '4.b': 'expand globally number of scholarships available developing countries enrolment higher education vocational training programmes',
    '4.c': 'increase supply qualified teachers including international cooperation for teacher training developing countries',
    # SDG 5 — Gender Equality (9 targets)
    '5.1': 'end all forms of discrimination against all women girls everywhere',
    '5.2': 'eliminate all forms violence against all women girls public private spheres trafficking sexual other types exploitation',
    '5.3': 'eliminate all harmful practices child early forced marriage female genital mutilation',
    '5.4': 'recognize value unpaid care domestic work provision public services infrastructure social protection policies promotion shared responsibility household',
    '5.5': 'ensure women full effective participation equal opportunities for leadership all levels decision-making political economic public life',
    '5.6': 'ensure universal access sexual reproductive health reproductive rights',
    '5.a': 'undertake reforms give women equal rights economic resources access ownership control over land other forms property financial services',
    '5.b': 'enhance use of enabling technology in particular information communications technology to promote empowerment of women',
    '5.c': 'adopt strengthen sound policies enforceable legislation promotion gender equality empowerment all women girls all levels',
    # SDG 6 — Clean Water and Sanitation (8 targets)
    '6.1': 'achieve universal equitable access to safe affordable drinking water for all',
    '6.2': 'achieve access to adequate equitable sanitation hygiene for all end open defecation special attention women girls vulnerable',
    '6.3': 'improve water quality reducing pollution eliminating dumping minimizing release hazardous chemicals materials halving untreated wastewater recycling reuse',
    '6.4': 'increase water-use efficiency across all sectors ensure sustainable withdrawals supply freshwater address water scarcity',
    '6.5': 'implement integrated water resources management at all levels including through transboundary cooperation',
    '6.6': 'protect restore water-related ecosystems mountains forests wetlands rivers aquifers lakes',
    '6.a': 'expand international cooperation capacity-building support to developing countries water sanitation activities programmes',
    '6.b': 'support strengthen participation local communities improving water sanitation management',
    # SDG 7 — Affordable and Clean Energy (5 targets)
    '7.1': 'ensure universal access affordable reliable modern energy services',
    '7.2': 'increase substantially share of renewable energy in global energy mix',
    '7.3': 'double global rate of improvement in energy efficiency',
    '7.a': 'enhance international cooperation facilitate access clean energy research technology renewable energy efficiency cleaner fossil-fuel',
    '7.b': 'expand infrastructure upgrade technology supplying modern sustainable energy services developing countries',
    # SDG 8 — Decent Work and Economic Growth (12 targets)
    '8.1': 'sustain per capita economic growth in accordance with national circumstances at least 7 per cent gross domestic product growth annum least developed',
    '8.2': 'achieve higher levels economic productivity through diversification technological upgrading innovation focus high-value added labour-intensive sectors',
    '8.3': 'promote development-oriented policies support productive activities decent job creation entrepreneurship creativity innovation formalization growth micro small medium enterprises',
    '8.4': 'improve global resource efficiency consumption production endeavour decouple economic growth environmental degradation',
    '8.5': 'achieve full productive employment decent work all women men young people persons disabilities equal pay work equal value',
    '8.6': 'reduce proportion youth not in employment education training',
    '8.7': 'eradicate forced labour end modern slavery human trafficking secure prohibition elimination worst forms child labour child soldiers',
    '8.8': 'protect labour rights promote safe secure working environments all workers including migrant workers women precarious employment',
    '8.9': 'devise implement policies promote sustainable tourism creates jobs promotes local culture products',
    '8.10': 'strengthen capacity domestic financial institutions encourage expand access to banking insurance financial services',
    '8.a': 'increase aid for trade support developing countries particularly least developed enhanced integrated framework trade-related technical assistance',
    '8.b': 'develop operationalize global strategy youth employment implement global jobs pact international labour organization',
    # SDG 9 — Industry Innovation Infrastructure (8 targets)
    '9.1': 'develop quality reliable sustainable resilient infrastructure regional transborder support economic development human well-being',
    '9.2': 'promote inclusive sustainable industrialization significantly raise share employment gross domestic product manufacturing',
    '9.3': 'increase access of small-scale industrial enterprises particularly developing countries financial services affordable credit integration value chains markets',
    '9.4': 'upgrade infrastructure retrofit industries make sustainable resource-use efficiency greater adoption clean environmentally sound technologies industrial processes',
    '9.5': 'enhance scientific research upgrade technological capabilities industrial sectors all countries developing encouraging innovation increasing research workers',
    '9.a': 'facilitate sustainable resilient infrastructure development developing countries enhanced financial technological technical support African landlocked island',
    '9.b': 'support domestic technology development research innovation developing countries ensuring conducive policy environment industrial diversification value addition',
    '9.c': 'significantly increase access information communications technology strive provide universal affordable access internet least developed countries',
    # SDG 10 — Reduced Inequalities (10 targets)
    '10.1': 'progressively achieve sustain income growth bottom 40 per cent of population at rate higher than national average',
    '10.2': 'empower promote social economic political inclusion all irrespective age sex disability race ethnicity origin religion economic status',
    '10.3': 'ensure equal opportunity reduce inequalities of outcome eliminating discriminatory laws policies practices promoting appropriate legislation',
    '10.4': 'adopt policies especially fiscal wage social protection progressively achieve greater equality',
    '10.5': 'improve regulation monitoring of global financial markets institutions strengthen implementation such regulations',
    '10.6': 'ensure enhanced representation voice for developing countries decision-making global international economic financial institutions',
    '10.7': 'facilitate orderly safe regular responsible migration mobility of people implementation planned well-managed migration policies',
    '10.a': 'implement principle of special differential treatment for developing countries particularly least developed accordance world trade organization',
    '10.b': 'encourage official development assistance financial flows including foreign direct investment to states where need is greatest',
    '10.c': 'reduce to less than 3 per cent transaction costs of migrant remittances eliminate remittance corridors with costs higher than 5 per cent',
    # SDG 11 — Sustainable Cities and Communities (10 targets)
    '11.1': 'ensure access for all to adequate safe affordable housing basic services upgrade slums',
    '11.2': 'provide access to safe affordable accessible sustainable transport systems all improving road safety expanding public transport',
    '11.3': 'enhance inclusive sustainable urbanization capacity participatory integrated sustainable human settlement planning management all countries',
    '11.4': 'strengthen efforts to protect safeguard worlds cultural natural heritage',
    '11.5': 'reduce number of deaths people affected substantially decrease direct economic losses caused by disasters water-related disasters',
    '11.6': 'reduce adverse per capita environmental impact of cities including paying special attention air quality municipal waste management',
    '11.7': 'provide universal access to safe inclusive accessible green public spaces particular for women children older persons disabilities',
    '11.a': 'support positive economic social environmental links between urban peri-urban rural areas strengthening national regional development planning',
    '11.b': 'increase number cities human settlements adopting implementing integrated policies plans towards inclusion resource efficiency adaptation climate change resilience disasters',
    '11.c': 'support least developed countries through financial technical assistance building sustainable resilient buildings utilizing local materials',
    # SDG 12 — Responsible Consumption and Production (11 targets)
    '12.1': 'implement 10-year framework programmes sustainable consumption production all countries developed countries taking lead',
    '12.2': 'achieve sustainable management efficient use of natural resources',
    '12.3': 'halve per capita global food waste at retail consumer levels reduce food losses along production supply chains including post-harvest',
    '12.4': 'achieve environmentally sound management of chemicals all wastes throughout life cycle accordance international frameworks reduce release air water soil',
    '12.5': 'substantially reduce waste generation through prevention reduction recycling reuse',
    '12.6': 'encourage companies especially large transnational adopt sustainable practices integrate sustainability information into reporting cycle',
    '12.7': 'promote public procurement practices that are sustainable in accordance national policies priorities',
    '12.8': 'ensure people everywhere have relevant information awareness for sustainable development lifestyles in harmony with nature',
    '12.a': 'support developing countries strengthen scientific technological capacity move towards more sustainable patterns consumption production',
    '12.b': 'develop implement tools to monitor sustainable development impacts for sustainable tourism creates jobs promotes local culture products',
    '12.c': 'rationalize inefficient fossil-fuel subsidies that encourage wasteful consumption removing market distortions accordance national circumstances',
    # SDG 13 — Climate Action (5 targets)
    '13.1': 'strengthen resilience adaptive capacity to climate-related hazards natural disasters all countries',
    '13.2': 'integrate climate change measures into national policies strategies planning',
    '13.3': 'improve education awareness-raising human institutional capacity climate change mitigation adaptation impact reduction early warning',
    '13.a': 'implement commitment undertaken by developed-country parties to united nations framework convention climate change goal mobilizing 100 billion annually green climate fund',
    '13.b': 'promote mechanisms for raising capacity for effective climate change-related planning management least developed countries small island developing states women youth local marginalized communities',
    # SDG 14 — Life Below Water (10 targets)
    '14.1': 'prevent significantly reduce marine pollution all kinds particular from land-based activities including marine debris nutrient pollution',
    '14.2': 'sustainably manage protect marine coastal ecosystems avoid significant adverse impacts strengthening resilience restore healthy productive oceans',
    '14.3': 'minimize address impacts of ocean acidification including through enhanced scientific cooperation all levels',
    '14.4': 'effectively regulate harvesting end overfishing illegal unreported unregulated fishing destructive fishing practices implement science-based management plans',
    '14.5': 'conserve at least 10 per cent of coastal marine areas consistent with national international law based on best available scientific information',
    '14.6': 'prohibit certain forms fisheries subsidies which contribute to overcapacity overfishing eliminate subsidies that contribute to illegal unreported unregulated fishing',
    '14.7': 'increase economic benefits to small island developing states least developed countries from sustainable use of marine resources fisheries aquaculture tourism',
    '14.a': 'increase scientific knowledge develop research capacity transfer marine technology improve ocean health enhance contribution marine biodiversity to development developing countries',
    '14.b': 'provide access for small-scale artisanal fishers to marine resources markets',
    '14.c': 'enhance conservation sustainable use of oceans their resources by implementing international law as reflected unclos law of the sea',
    # SDG 15 — Life on Land (12 targets)
    '15.1': 'ensure conservation restoration sustainable use of terrestrial inland freshwater ecosystems their services particular forests wetlands mountains drylands',
    '15.2': 'promote implementation sustainable management of all types of forests halt deforestation restore degraded forests substantially increase afforestation reforestation',
    '15.3': 'combat desertification restore degraded land soil including land affected by desertification drought floods strive achieve a land degradation-neutral world',
    '15.4': 'ensure conservation of mountain ecosystems including their biodiversity in order to enhance their capacity provide benefits essential for sustainable development',
    '15.5': 'take urgent significant action to reduce degradation of natural habitats halt loss of biodiversity protect prevent extinction of threatened species',
    '15.6': 'promote fair equitable sharing of benefits arising from utilization of genetic resources promote appropriate access such resources',
    '15.7': 'take urgent action to end poaching trafficking of protected species of flora fauna address both demand supply illegal wildlife products',
    '15.8': 'introduce measures to prevent introduction significantly reduce impact invasive alien species on land water ecosystems control or eradicate priority species',
    '15.9': 'integrate ecosystem biodiversity values into national local planning development processes poverty reduction strategies accounts',
    '15.a': 'mobilize significantly increase financial resources from all sources to conserve sustainably use biodiversity ecosystems',
    '15.b': 'mobilize significant resources from all sources at all levels to finance sustainable forest management provide adequate incentives developing countries advance such management conservation reforestation',
    '15.c': 'enhance global support for efforts to combat poaching trafficking of protected species including by increasing capacity local communities pursue sustainable livelihood opportunities',
    # SDG 16 — Peace Justice and Strong Institutions (12 targets)
    '16.1': 'significantly reduce all forms of violence related death rates everywhere',
    '16.2': 'end abuse exploitation trafficking all forms of violence against torture children',
    '16.3': 'promote rule of law at national international levels ensure equal access justice for all',
    '16.4': 'significantly reduce illicit financial arms flows strengthen recovery return stolen assets combat all forms organized crime',
    '16.5': 'substantially reduce corruption bribery in all their forms',
    '16.6': 'develop effective accountable transparent institutions all levels',
    '16.7': 'ensure responsive inclusive participatory representative decision-making at all levels',
    '16.8': 'broaden strengthen participation of developing countries in institutions of global governance',
    '16.9': 'provide legal identity for all including birth registration',
    '16.10': 'ensure public access to information protect fundamental freedoms accordance national legislation international agreements',
    '16.a': 'strengthen relevant national institutions including through international cooperation building capacity all levels particular developing countries prevent violence combat terrorism crime',
    '16.b': 'promote enforce non-discriminatory laws policies for sustainable development',
    # SDG 17 — Partnerships for the Goals (19 targets)
    '17.1': 'strengthen domestic resource mobilization including through international support to developing countries to improve domestic capacity tax other revenue collection',
    '17.2': 'developed countries to implement fully their official development assistance commitments 0.7 per cent of gross national income developing countries 0.15 0.20 per cent least developed countries',
    '17.3': 'mobilize additional financial resources for developing countries from multiple sources',
    '17.4': 'assist developing countries in attaining long-term debt sustainability through coordinated policies aimed at fostering debt financing debt relief debt restructuring as appropriate address external debt highly indebted poor countries',
    '17.5': 'adopt implement investment promotion regimes for least developed countries',
    '17.6': 'enhance north-south south-south triangular regional international cooperation on access to science technology innovation enhance knowledge sharing',
    '17.7': 'promote development transfer dissemination diffusion of environmentally sound technologies to developing countries on favourable terms including concessional preferential terms',
    '17.8': 'fully operationalize technology bank science technology innovation capacity-building mechanism for least developed countries enhance use of enabling technology in particular information communications technology',
    '17.9': 'enhance international support implementing effective targeted capacity-building developing countries support national plans implement all sustainable development goals including through north-south south-south triangular cooperation',
    '17.10': 'promote universal rules-based open non-discriminatory equitable multilateral trading system under world trade organization including through conclusion negotiations doha development agenda',
    '17.11': 'significantly increase exports of developing countries particular view to doubling least developed countries share of global exports by 2020',
    '17.12': 'realize timely implementation of duty-free quota-free market access on a lasting basis for all least developed countries consistent with world trade organization decisions',
    '17.13': 'enhance global macroeconomic stability including through policy coordination policy coherence',
    '17.14': 'enhance policy coherence for sustainable development',
    '17.15': 'respect each countrys policy space leadership to establish implement policies for poverty eradication sustainable development',
    '17.16': 'enhance global partnership for sustainable development complemented by multi-stakeholder partnerships that mobilize share knowledge expertise technology financial resources support achievement of sustainable development goals all countries particularly developing',
    '17.17': 'encourage promote effective public public-private civil society partnerships building on the experience resourcing strategies of partnerships',
    '17.18': 'enhance capacity-building support to developing countries including least developed countries small island developing states to increase significantly availability of high-quality timely reliable data disaggregated by income gender age race ethnicity migratory status disability geographic location other characteristics relevant in national contexts',
    '17.19': 'build on existing initiatives to develop measurements of progress on sustainable development that complement gross domestic product support statistical capacity-building developing countries',
}

assert len(TARGET_DESC) == 169, f'Expected 169, got {len(TARGET_DESC)}'
print(f'Defined TARGET_DESC for {len(TARGET_DESC)} targets')

# Now save to disk so future runs can pick it up via the existing fallback
import json, os
out_path = os.path.join(DATA_CACHE, 'target_descriptors_169.json')
with open(out_path, 'w') as f:
    json.dump(TARGET_DESC, f, indent=2)
print(f'Saved to {out_path}')

In [ ]:
import json
with open(f'{DATA_CACHE}/target_descriptors_169.json', 'w') as f:
    json.dump(TARGET_DESC, f, indent=2)
print(f'Saved {len(TARGET_DESC)} target descriptors')

**Cell A — entity evidence + three-way combined**

In [ ]:
# ── doc_entities evidence + three-way combined (LLM prior + neighbours + entities) ──
# Builds a per-paper entity-vs-target overlap matrix, then calibrates a
# three-way weighting (λ_neighbours, λ_entities) on val for each model.
import re

# Target keyword sets (lifted from per-run notebook's target descriptor list)
TARGET_DESC_PATH = os.path.join(DATA_CACHE, 'target_descriptors_169.json')
if os.path.exists(TARGET_DESC_PATH):
    with open(TARGET_DESC_PATH) as f:
        TARGET_DESC = json.load(f)
else:
    print('No target_descriptors_169.json — falling back to TARGET_IDS-only keywords.')
    TARGET_DESC = {tid: tid for tid in TARGET_IDS}

def _kw(s):
    return {w.lower() for w in re.findall(r'\b[a-zA-Z]{4,}\b', s)}

target_kws_169 = [_kw(TARGET_DESC.get(tid, tid)) for tid in TARGET_IDS]

# Per-paper evidence: count of overlapping keywords per goal (max-pool from targets to goals)
def entity_goal_evidence(entities_list):
    n = len(entities_list)
    out_t = np.zeros((n, 169), dtype=np.float32)   # at target level first
    for i, ents in enumerate(entities_list):
        ents_l = {e.lower() for e in ents if isinstance(e, str)}
        if not ents_l: continue
        for j in range(169):
            tk = target_kws_169[j]
            if tk:
                out_t[i, j] = len(ents_l & tk) / max(len(tk), 1)
    # Pool to goal level (max over targets of each goal)
    out = np.zeros((n, 17), dtype=np.float32)
    for g in range(1, 18):
        idxs = GOAL_TO_TARGETS[g]
        out[:, g-1] = out_t[:, idxs].max(axis=1)
    # Normalize per-paper to [0,1] so it sits on same scale as prior
    row_max = out.max(axis=1, keepdims=True); row_max[row_max == 0] = 1.0
    return out / row_max

print('Computing entity evidence on val + test...')
ent_evid_val  = entity_goal_evidence(entities_val)
ent_evid_test = entity_goal_evidence(entities_test)
print(f'  val evidence: {ent_evid_val.shape}, density>0: {(ent_evid_val>0).mean():.3f}')
print(f'  test evidence: {ent_evid_test.shape}, density>0: {(ent_evid_test>0).mean():.3f}')

# Best K from the ablation (per model) — re-use evid_per_k_*
best_K_per_model = (df_abl[df_abl['lambda']=='global']
                     .sort_values('test_f1', ascending=False)
                     .groupby('model').first())['K'].to_dict()
print(f'\nBest K per model (from global-λ branch): {best_K_per_model}')

# Three-way calibration: grid search (λ_n, λ_e) on val
def calibrate_three_way(prior_v, evid_n_v, evid_e_v, labels_v):
    grid = np.arange(0.0, 1.01, 0.1)
    best_lams, best_f1 = (0.0, 0.0), -1.0
    for ln in grid:
        for le in grid:
            if ln + le > 1.0 + 1e-9: continue
            combo = (1 - ln - le) * prior_v + ln * evid_n_v + le * evid_e_v
            thr = opt_threshold_per_class(combo, labels_v)
            f = macro_f1(combo, labels_v, thr)
            if f > best_f1:
                best_f1 = f
                best_lams = (ln, le)
    return best_lams, best_f1

three_way_rows = []
for m in MODELS:
    g_t = cached[m]['goal_test']; g_v = cached[m]['goal_val']
    K = best_K_per_model[m]
    evid_n_v, evid_n_t = evid_per_k_val[K], evid_per_k_test[K]

    (ln, le), f1_v = calibrate_three_way(g_v, evid_n_v, ent_evid_val,
                                          labels_val.astype(np.float32))
    combo_v = (1-ln-le) * g_v + ln * evid_n_v + le * ent_evid_val
    combo_t = (1-ln-le) * g_t + ln * evid_n_t + le * ent_evid_test
    thr = opt_threshold_per_class(combo_v, labels_val)
    f1_t = macro_f1(combo_t, labels_test, thr)

    three_way_rows.append({
        'model': m, 'K_neighbours': K,
        'lambda_neighbours': ln, 'lambda_entities': le,
        'lambda_prior':      round(1-ln-le, 2),
        'val_f1':  round(f1_v, 4),
        'test_f1': round(f1_t, 4),
    })

df_three = pd.DataFrame(three_way_rows).set_index('model')
print('\n── THREE-WAY (LLM prior + neighbours + entities) at best K ──')
print(df_three.to_string())

**Cell B — McNemar significance test**

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar
print(mcnemar)   # should print: <function mcnemar at 0x...>

In [ ]:
# ── McNemar's test: is Gemma > Mistral statistically? ──
# Imports first, before any usage.
try:
    from statsmodels.stats.contingency_tables import mcnemar
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'statsmodels'], check=True)
    from statsmodels.stats.contingency_tables import mcnemar

import numpy as np

# For each model, get best-config test predictions
def best_predictions_per_model():
    preds = {}
    for m in MODELS:
        row = df_three.loc[m]
        ln, le = row['lambda_neighbours'], row['lambda_entities']
        K = int(row['K_neighbours'])
        g_t = cached[m]['goal_test']
        g_v = cached[m]['goal_val']
        combo_v = (1-ln-le) * g_v + ln * evid_per_k_val[K]  + le * ent_evid_val
        combo_t = (1-ln-le) * g_t + ln * evid_per_k_test[K] + le * ent_evid_test
        thr = opt_threshold_per_class(combo_v, labels_val)
        preds[m] = (combo_t >= thr).astype(int)
    return preds

preds_per_model = best_predictions_per_model()

# McNemar on per-(paper, goal) correctness
print('\n── McNemar pairwise tests (per-(paper,goal) correctness, two-sided) ──')
print(f'{"pair":40s} {"b":>6s} {"c":>6s} {"χ²":>8s} {"p":>10s} {"verdict"}')
print('-' * 90)

flat_labels = labels_test.flatten()
for ma in MODELS:
    correct_a = (preds_per_model[ma].flatten() == flat_labels).astype(int)
    for mb in MODELS:
        if mb <= ma: continue
        correct_b = (preds_per_model[mb].flatten() == flat_labels).astype(int)
        b = int(((correct_a == 1) & (correct_b == 0)).sum())
        c = int(((correct_a == 0) & (correct_b == 1)).sum())
        if b + c == 0:
            print(f'{ma+" vs "+mb:40s} {b:>6d} {c:>6d} {"--":>8s} {"--":>10s}  identical')
            continue
        result = mcnemar([[0, b], [c, 0]], exact=False, correction=True)
        verdict = ('A>B sig' if (b > c and result.pvalue < 0.05) else
                   'B>A sig' if (c > b and result.pvalue < 0.05) else
                   'tie')
        print(f'{ma+" vs "+mb:40s} {b:>6d} {c:>6d} {result.statistic:>8.2f} '
              f'{result.pvalue:>10.2e}  {verdict}')

**Cell C — per-goal F1 breakdown + final verdict**

In [ ]:
# ── Per-goal F1 at best three-way config + final paper-ready summary ──
goal_names = {1:'No Poverty', 2:'Zero Hunger', 3:'Health', 4:'Education',
              5:'Gender', 6:'Water', 7:'Energy', 8:'Decent Work',
              9:'Innovation', 10:'Inequality', 11:'Cities', 12:'Consumption',
              13:'Climate', 14:'Sea', 15:'Land', 16:'Peace', 17:'Partnership'}

per_goal_f1 = {}
for m in MODELS:
    p = preds_per_model[m]
    # macro f1 per goal
    per_goal_f1[m] = [f1_score(labels_test[:, g], p[:, g], zero_division=0)
                      for g in range(17)]

df_goals = pd.DataFrame(per_goal_f1, index=[f'SDG{g+1} {goal_names[g+1]}' for g in range(17)])
df_goals['best_model'] = df_goals.idxmax(axis=1)
df_goals['best_f1']    = df_goals[MODELS].max(axis=1)
df_goals['gap_to_2nd'] = df_goals[MODELS].apply(
    lambda row: row.nlargest(2).iloc[0] - row.nlargest(2).iloc[1], axis=1)

print('\n── PER-GOAL F1 at best three-way config ──')
print(df_goals.round(4).to_string())

print('\n── Goals where each model is the best ──')
for m in MODELS:
    won = df_goals[df_goals['best_model'] == m]
    print(f'  {m:15s}  wins on {len(won)} goals: {sorted(won.index.tolist())}')

# Final paper-ready summary line per model
print('\n' + '='*72)
print('PAPER-READY SUMMARY')
print('='*72)
for m in MODELS:
    row = df_three.loc[m]
    print(f'  {m:15s}  best F1={row["test_f1"]:.4f}  '
          f'(λ_prior={row["lambda_prior"]:.1f}, '
          f'λ_neighbours={row["lambda_neighbours"]:.1f}, '
          f'λ_entities={row["lambda_entities"]:.1f}, '
          f'K={row["K_neighbours"]})')

# Save everything to the v3 output dir
out_dir = os.path.join(DRIVE_ROOT, 'cross_model_comparison_v5')
df_three.to_csv(os.path.join(out_dir, 'three_way_combined.csv'))
df_goals.to_csv(os.path.join(out_dir, 'per_goal_f1.csv'))
print(f'\nSaved: {out_dir}/three_way_combined.csv')
print(f'       {out_dir}/per_goal_f1.csv')


**Cell A — entity diagnostic**

In [ ]:
# Diagnostic: WHY do entities not help?
# 1. Match-rate analysis: how often does any entity overlap any target keyword?
# 2. Per-goal: is the entity contribution concentrated where the LLM was already confident?

import re
from sklearn.metrics import f1_score

def _kw(s):
    return {w.lower() for w in re.findall(r'\b[a-zA-Z]{4,}\b', s)}

target_kws_169 = [_kw(TARGET_DESC.get(tid, tid)) for tid in TARGET_IDS]
goal_kws = {g: set().union(*[target_kws_169[i] for i in GOAL_TO_TARGETS[g]])
            for g in range(1, 18)}

# Per-paper, per-goal: did ANY entity match the goal vocabulary?
match_rate = np.zeros((len(entities_test), 17), dtype=np.float32)
for i, ents in enumerate(entities_test):
    ents_l = {str(e).lower() for e in ents}
    if not ents_l: continue
    for g in range(1, 18):
        if ents_l & goal_kws[g]:
            match_rate[i, g-1] = 1.0

print('=== Entity-vocabulary match diagnostic ===')
print(f'Mean match rate per (paper, goal): {match_rate.mean():.3f}')
print(f'  (i.e., for a random paper-goal pair, only this fraction has any entity-vocab overlap)')
print(f'Per-goal match rate (top 5):  {sorted(match_rate.mean(axis=0).round(3), reverse=True)[:5]}')
print(f'Per-goal match rate (bot 5):  {sorted(match_rate.mean(axis=0).round(3))[:5]}')

# Where the LLM was already confident, does entity evidence agree?
# (If it does → redundant. If it doesn't → could help, but doesn't.)
g_t = cached['gemma4-26b']['goal_test']
ent_evid = ent_evid_test  # already computed in Cell 21

# For each goal, correlation between LLM prior and entity evidence
print('\n=== Per-goal correlation: LLM prior ↔ entity evidence (Gemma) ===')
print('High correlation → entity evidence is redundant with what LLM already encodes')
for g in range(17):
    if ent_evid[:, g].std() > 0:
        corr = np.corrcoef(g_t[:, g], ent_evid[:, g])[0, 1]
        print(f'  SDG {g+1:2d}: r = {corr:+.3f}  '
              f'(λ_ent on this goal in best three-way: {df_three.loc["gemma4-26b"]["lambda_entities"]:.1f})')
    else:
        print(f'  SDG {g+1:2d}: entity evidence is all zero')

**Cell B — graph diagnostic**

In [ ]:
# Replacement Cell B — diagnostic: WHY does graph propagation not help?
# Hypothesis: the LLM's Stage 2 probabilities already encode cross-target
# correlations similar to those in Pradhan/SBERT, making explicit propagation
# redundant. We measure this directly.
import os

# 1. Load matrix C (SBERT) — saved on disk by the per-run notebooks
SBERT_PATH = os.path.join(DATA_CACHE, 'sbert_target_matrix_169.npy')
if os.path.exists(SBERT_PATH):
    W_C = np.load(SBERT_PATH)
    print(f'Loaded W_C (SBERT 169x169) from {SBERT_PATH}, shape {W_C.shape}')
else:
    print(f'Missing {SBERT_PATH} — re-encoding 169 target descriptors with SBERT...')
    target_desc_list = [TARGET_DESC[tid] for tid in TARGET_IDS]
    target_emb = sbert.encode(target_desc_list, normalize_embeddings=True,
                               batch_size=64, show_progress_bar=False).astype(np.float32)
    W_C = target_emb @ target_emb.T
    np.fill_diagonal(W_C, 0.0)
    print(f'Built W_C: {W_C.shape}')

# 2. Build matrix B (Pradhan inherited) inline from the JSON.
# 2. Build matrix B (Pradhan inherited) from the JSON.
PRADHAN_JSON = os.path.join(GRAPH_DIR,
                              'sdg_interaction_matrix_v3_2.json')
if os.path.exists(PRADHAN_JSON):
    with open(PRADHAN_JSON) as f:
        pradhan = json.load(f)
    # The JSON stores the 17x17 goal-level matrix as a nested list under "matrix_W"
    # (with metadata, labels, signed version, and pair counts under sibling keys).
    W_A = np.array(pradhan['matrix_W'], dtype=np.float32)
    np.fill_diagonal(W_A, 0.0)
    print(f'Loaded W_A (Pradhan goal-level 17x17) from JSON, range '
          f'[{W_A.min():+.3f}, {W_A.max():+.3f}], nnz={int((W_A != 0).sum())}')

    # Inherit goal-level matrix to target level
    W_B = np.zeros((169, 169), dtype=np.float32)
    goal_of = {i: g for g in range(1, 18) for i in GOAL_TO_TARGETS[g]}
    for i in range(169):
        for j in range(169):
            gi, gj = goal_of[i], goal_of[j]
            if gi != gj:
                W_B[i, j] = W_A[gi - 1, gj - 1]
    print(f'Built W_B (Pradhan inherited 169x169), nnz={int((W_B != 0).sum())}')
else:
    W_B = None
    print(f'Pradhan JSON not at {PRADHAN_JSON} — Pradhan comparison will be skipped')

# 3. Empirical cross-target correlation from Gemma's Stage 2 outputs
target_probs = cached['gemma4-26b']['target_test']
emp_corr = np.corrcoef(target_probs.T)
np.fill_diagonal(emp_corr, 0.0)
print(f'Empirical correlation matrix from Gemma target_probs: {emp_corr.shape}')

# 4. Compare matrices on the off-diagonal upper triangle
def matrix_similarity(A, B):
    iu = np.triu_indices(A.shape[0], k=1)
    a, b = A[iu], B[iu]
    if a.std() == 0 or b.std() == 0: return float('nan')
    return float(np.corrcoef(a, b)[0, 1])

print('\n=== Cross-target correlation already encoded by the LLM ===')
print('If the LLM has absorbed SDG-correlation structure during pretraining,')
print('the empirical correlation of its target probabilities should resemble')
print('Pradhan and/or SBERT matrices — making explicit graph propagation redundant.')
print()
print(f'Pearson r(empirical-corr, SBERT-semantic C):       '
      f'{matrix_similarity(emp_corr, W_C):+.3f}')
if W_B is not None:
    print(f'Pearson r(empirical-corr, Pradhan-inherited B):    '
          f'{matrix_similarity(emp_corr, W_B):+.3f}')

# 5. Stronger version: top-K strongest pairs overlap
print()
top_k = 50
emp_top = set(zip(*np.unravel_index(
    np.argsort(np.abs(emp_corr), axis=None)[-top_k:], emp_corr.shape)))
sbert_top = set(zip(*np.unravel_index(
    np.argsort(np.abs(W_C), axis=None)[-top_k:], W_C.shape)))
overlap_C = len(emp_top & sbert_top) / top_k
print(f'Top-{top_k} strongest target pairs:')
print(f'  {overlap_C*100:.0f}% overlap between Gemma empirical and SBERT-semantic')
if W_B is not None:
    pradhan_top = set(zip(*np.unravel_index(
        np.argsort(np.abs(W_B), axis=None)[-top_k:], W_B.shape)))
    overlap_B = len(emp_top & pradhan_top) / top_k
    print(f'  {overlap_B*100:.0f}% overlap between Gemma empirical and Pradhan-inherited')


In [ ]:
# Inspect what's actually in the Pradhan JSON
import json, os
PRADHAN_JSON = os.path.join(GRAPH_DIR,
                              'sdg_interaction_matrix_v3_2.json')
print(f'Path: {PRADHAN_JSON}')
print(f'Exists: {os.path.exists(PRADHAN_JSON)}')
if os.path.exists(PRADHAN_JSON):
    with open(PRADHAN_JSON) as f:
        pradhan = json.load(f)
    print(f'Type: {type(pradhan).__name__}')
    print(f'Length: {len(pradhan)}')
    items = list(pradhan.items()) if isinstance(pradhan, dict) else list(enumerate(pradhan))
    print(f'First 5 items:')
    for k, v in items[:5]:
        print(f'  {k!r}: {v!r}')


**Aurora label-cooccurrence graph**

In [ ]:
# =============================================================================
# CELL — Experiment 1: Aurora label-cooccurrence graph
# =============================================================================

# We test the prediction implicit in the redundancy diagnostic: a graph
# that's (a) NOVEL relative to LLM training AND (b) ALIGNED with the
# labelled corpus should beat the seven graphs in the paper.
#
# The neighbour-pool labels (df_pool, ~3000 docs disjoint from train/val/test)
# satisfy both conditions. Pradhan satisfies (a) but not (b); SBERT
# satisfies (b) approximately but not (a). Aurora label co-occurrence
# satisfies both by construction.
#
# We build the graph AT GOAL LEVEL (17x17, dense) to keep the matrix
# meaningful — at target level, the pool's per-target positive counts
# are too sparse to produce a stable correlation matrix.

import numpy as np
from sklearn.metrics import f1_score

# Need the propagation operator and the per-class-threshold helper.
# Both are already in scope if you ran Cells 11 + 25 earlier; redefine
# defensively in case the kernel was restarted between sections.
def propagate_gated_signed(probs, W, alpha=0.8, conf=0.6):
    Wp = np.where(W > 0, W, 0).copy()
    Wn = np.abs(np.where(W < 0, W, 0)).copy()
    rp = Wp.sum(1, keepdims=True); rp[rp == 0] = 1.0; Wp /= rp
    rn = Wn.sum(1, keepdims=True); rn[rn == 0] = 1.0; Wn /= rn
    gate = (probs >= conf).astype(float)
    gated = probs * gate
    return np.clip(alpha * probs + (1 - alpha) * (gated @ Wp.T)
                   - (1 - alpha) * (gated @ Wn.T), 0, 1)

# ── Build the Aurora goal-level co-occurrence graph from the neighbour pool ──
# Empirical lift: P(both | either) - P(i)P(j). Centred so synergies are
# positive, anti-correlations negative. Symmetric, zero diagonal.
labels_pool = pool_labels_goal  # (N_pool, 17) already in scope
N = labels_pool.shape[0]
P_i = labels_pool.mean(axis=0)                                       # (17,)
P_ij = (labels_pool.T @ labels_pool) / N                             # (17, 17)
# P(both | either) = P(both) / P(either) = P(both) / (P(i) + P(j) - P(both))
P_either = P_i[:, None] + P_i[None, :] - P_ij
P_both_given_either = np.where(P_either > 0, P_ij / P_either, 0)
# Lift: how much more often than expected do these goals co-occur
W_aurora_17 = P_both_given_either - P_i[:, None] * P_i[None, :]
np.fill_diagonal(W_aurora_17, 0.0)

print(f'Aurora-cooccurrence W (17x17), range '
      f'[{W_aurora_17.min():+.3f}, {W_aurora_17.max():+.3f}], '
      f'pos pairs={int((W_aurora_17 > 0).sum())}, '
      f'neg pairs={int((W_aurora_17 < 0).sum())}')

# ── Evaluate it through the existing propagation operator ──
# We propagate at goal level (g_t and g_v are (N, 17)) and tune
# (alpha, conf) on val, exactly the recipe the per-run notebook uses
# for the seven matrices.

def opt_threshold_per_class(probs, labels, grid=None):
    if grid is None:
        grid = np.arange(0.05, 0.96, 0.05)
    C = probs.shape[1]
    thr = np.zeros(C, dtype=np.float32)
    for c in range(C):
        best_f1, best_t = -1.0, 0.5
        for t in grid:
            preds = (probs[:, c] >= t).astype(int)
            f = f1_score(labels[:, c], preds, zero_division=0)
            if f > best_f1: best_f1, best_t = f, t
        thr[c] = best_t
    return thr

def macro_f1(probs, labels, thr):
    preds = (probs >= thr).astype(int)
    return f1_score(labels, preds, average='macro', zero_division=0)

ALPHA_GRID = [0.7, 0.8, 0.85, 0.9, 0.95]
CONF_GRID  = [0.5, 0.6, 0.7]

# We need raw test/val goal probabilities for each model. They're in `cached`.
print('\n=== Experiment 1: Aurora-cooccurrence graph propagation ===')
print(f'{"Model":15s}  {"best alpha":>10s} {"best conf":>10s}  {"val F1":>8s}  {"test F1":>8s}  {"vs B2":>8s}')
print('-' * 80)

aurora_graph_results = {}
for m in MODELS:
    g_t = cached[m]['goal_test']  # (10000, 17)
    g_v = cached[m]['goal_val']   # (200, 17)

    # Baseline B2: target-aggregated probs as the prior — same as in the paper
    # For this experiment we propagate Stage 1 goal probs g_t directly, since
    # the matrix is goal-level. (Paper Matrix A uses the same convention.)
    best = (None, None, -1.0, -1.0)
    for alpha in ALPHA_GRID:
        for conf in CONF_GRID:
            prop_v = propagate_gated_signed(g_v, W_aurora_17, alpha=alpha, conf=conf)
            prop_t = propagate_gated_signed(g_t, W_aurora_17, alpha=alpha, conf=conf)
            thr = opt_threshold_per_class(prop_v, labels_val)
            f1_v = macro_f1(prop_v, labels_val, thr)
            f1_t = macro_f1(prop_t, labels_test, thr)
            if f1_v > best[2]:
                best = (alpha, conf, f1_v, f1_t)

    # Compute B2 baseline for this model for the delta
    thr_b2 = opt_threshold_per_class(g_v, labels_val)
    f1_b2  = macro_f1(g_t, labels_test, thr_b2)
    delta  = best[3] - f1_b2
    aurora_graph_results[m] = {
        'alpha': best[0], 'conf': best[1],
        'val_f1': best[2], 'test_f1': best[3],
        'b2_f1': f1_b2, 'delta_b2': delta,
    }
    print(f'{m:15s}  {best[0]:>10.2f} {best[1]:>10.1f}  '
          f'{best[2]:>8.4f}  {best[3]:>8.4f}  {delta:>+8.4f}')

# ── Comparison: this graph vs. the seven paper graphs vs. doc_neighbours ──
# For the four backbones, what's the best graph result reported in the paper?
# We use Gemma's number from the paper's Table 5 as the reference: the val-
# best graph (matrix F, Keyword overlap) reaches test F1 = 0.5215, which is
# +0.0036 over the no-graph baseline B1 and -0.0131 below B2.

print('\n--- Comparison summary (Gemma backbone, the paper\'s primary) ---')
gemma_aurora = aurora_graph_results['gemma4-26b']
print(f'  Paper\'s best graph (matrix F, Keyword overlap):  test F1 = 0.5215  '
      f'(Δ vs B1 = +0.0036)')
print(f'  Aurora-cooccurrence (this experiment):           '
      f'test F1 = {gemma_aurora["test_f1"]:.4f}  '
      f'(Δ vs B2 = {gemma_aurora["delta_b2"]:+.4f})')
print(f'  doc_neighbours best (K=50, global λ):            test F1 = 0.6117  '
      f'(Δ vs B2 = +0.0732)')

# Save for the paper
import os, pandas as pd
out_dir = os.path.join(DRIVE_ROOT, 'cross_model_comparison_v5')
os.makedirs(out_dir, exist_ok=True)
df_aurora = pd.DataFrame(aurora_graph_results).T
df_aurora.to_csv(os.path.join(out_dir, 'aurora_cooccurrence_graph.csv'))
print(f'\nSaved: {out_dir}/aurora_cooccurrence_graph.csv')


**Per-goal lift versus class prevalence**

In [ ]:
# =============================================================================
# CELL — Experiment 2: per-goal lift versus class prevalence
# =============================================================================

# Hypothesis: doc_neighbours wins specifically because rare classes benefit
# disproportionately. We test it directly.
#
# For each goal, compute:
#   - prevalence in the test set: P(y=1)
#   - per-goal F1 under B2 (target-LLM aggregated)
#   - per-goal F1 under doc_neighbours at the per-backbone optimum
#   - lift = doc_neighbours F1 - B2 F1
#
# Then regress lift on prevalence. A strongly negative slope confirms the
# hypothesis.

import numpy as np
from sklearn.metrics import f1_score

def aggregate_targets_to_goals(target_probs):
    out = np.zeros((target_probs.shape[0], 17), dtype=np.float32)
    for g in range(1, 18):
        idxs = GOAL_TO_TARGETS[g]
        out[:, g-1] = target_probs[:, idxs].max(axis=1)
    return out

# Per-goal positive prevalence in the test set
test_prevalence = labels_test.mean(axis=0)   # (17,)
print(f'Test-set positive prevalence per goal:')
print(f'  range [{test_prevalence.min():.3f}, {test_prevalence.max():.3f}]')
print(f'  median {np.median(test_prevalence):.3f}')

# Compute per-goal F1 under B2 and under doc_neighbours per backbone
prev_lift_rows = []
for m in MODELS:
    row = df_three.loc[m]
    K = int(row['K_neighbours'])
    g_t = cached[m]['goal_test']; g_v = cached[m]['goal_val']
    t_t = cached[m]['target_test']; t_v = cached[m]['target_val']

    # B2 baseline: target-LLM aggregated, per-class threshold on val
    g_t_agg = aggregate_targets_to_goals(t_t)
    g_v_agg = aggregate_targets_to_goals(t_v)
    thr_b2 = opt_threshold_per_class(g_v_agg, labels_val)
    preds_b2 = (g_t_agg >= thr_b2).astype(int)

    # doc_neighbours at backbone-optimum (global lambda from the ablation table)
    # Look up best lambda on val for this (m, K)
    sub = df_abl[(df_abl['model']==m) & (df_abl['K']==K) & (df_abl['lambda']=='global')]
    if len(sub):
        lam = float(sub['mean_lambda'].iloc[0])
    else:
        lam = 0.5  # fallback if exact (m, K) row missing
    evid_v = evid_per_k_val[K]; evid_t = evid_per_k_test[K]
    combo_v = (1 - lam) * g_v + lam * evid_v
    combo_t = (1 - lam) * g_t + lam * evid_t
    thr_dn = opt_threshold_per_class(combo_v, labels_val)
    preds_dn = (combo_t >= thr_dn).astype(int)

    for g_idx in range(17):
        f1_b2_g = f1_score(labels_test[:, g_idx], preds_b2[:, g_idx], zero_division=0)
        f1_dn_g = f1_score(labels_test[:, g_idx], preds_dn[:, g_idx], zero_division=0)
        prev_lift_rows.append({
            'model':      m,
            'goal':       g_idx + 1,
            'prevalence': float(test_prevalence[g_idx]),
            'b2_f1':      float(f1_b2_g),
            'docnbr_f1':  float(f1_dn_g),
            'lift':       float(f1_dn_g - f1_b2_g),
        })

df_prev = pd.DataFrame(prev_lift_rows)
print(f'\n=== Experiment 2: per-goal lift vs. test-prevalence ===')
print('\nPer-backbone slope of (lift vs. prevalence) — negative slope means rare goals benefit more:')
print(f'{"backbone":15s}  {"slope":>10s} {"intercept":>10s}  {"r":>7s}  {"rare-goal mean lift":>20s}  {"common-goal mean lift":>22s}')
print('-' * 100)
for m in MODELS:
    sub = df_prev[df_prev['model'] == m]
    x, y = sub['prevalence'].values, sub['lift'].values
    if x.std() > 0:
        slope, intercept = np.polyfit(x, y, 1)
        r = np.corrcoef(x, y)[0, 1]
    else:
        slope = intercept = r = float('nan')
    # split into rare vs common (median split on test prevalence)
    median_prev = np.median(x)
    rare_mean   = sub[sub['prevalence'] <= median_prev]['lift'].mean()
    common_mean = sub[sub['prevalence']  > median_prev]['lift'].mean()
    print(f'{m:15s}  {slope:>+10.3f} {intercept:>+10.4f}  {r:>+7.3f}  '
          f'{rare_mean:>+20.4f}  {common_mean:>+22.4f}')

# Across all 4 backbones combined
x_all = df_prev['prevalence'].values
y_all = df_prev['lift'].values
if x_all.std() > 0:
    slope_all, intercept_all = np.polyfit(x_all, y_all, 1)
    r_all = np.corrcoef(x_all, y_all)[0, 1]
    print(f'\nPooled across all 4 backbones (n={len(df_prev)}):')
    print(f'  slope = {slope_all:+.3f},  intercept = {intercept_all:+.4f},  '
          f'Pearson r = {r_all:+.3f}')

# Save for the paper
df_prev.to_csv(os.path.join(out_dir, 'prevalence_lift.csv'), index=False)
print(f'\nSaved: {out_dir}/prevalence_lift.csv')
print(f'Read it for a per-(model, goal) breakdown of B2 F1, doc_neighbours F1, and the lift.')

In [ ]:
# =============================================================================
# CELL — Experiment 5 (v2): WHY does doc_neighbours work? Flip analysis.
# =============================================================================

import numpy as np
from sklearn.metrics import f1_score

def aggregate_targets_to_goals(target_probs):
    out = np.zeros((target_probs.shape[0], 17), dtype=np.float32)
    for g in range(1, 18):
        idxs = GOAL_TO_TARGETS[g]
        out[:, g-1] = target_probs[:, idxs].max(axis=1)
    return out

m = 'gemma4-26b'
g_t = cached[m]['goal_test']; g_v = cached[m]['goal_val']
t_t = cached[m]['target_test']; t_v = cached[m]['target_val']

K_best   = int(df_three.loc[m]['K_neighbours'])
lam_best = float(df_abl[(df_abl['model']==m) & (df_abl['K']==K_best) &
                          (df_abl['lambda']=='global')]['mean_lambda'].iloc[0])
print(f"Using Gemma's post-ablation optimum: K={K_best}, global lambda={lam_best:.2f}")

g_t_agg = aggregate_targets_to_goals(t_t)
g_v_agg = aggregate_targets_to_goals(t_v)
thr_b2 = opt_threshold_per_class(g_v_agg, labels_val)
preds_b2 = (g_t_agg >= thr_b2).astype(int)

evid_v = evid_per_k_val[K_best]; evid_t = evid_per_k_test[K_best]
combo_v = (1 - lam_best) * g_v + lam_best * evid_v
combo_t = (1 - lam_best) * g_t + lam_best * evid_t
thr_dn = opt_threshold_per_class(combo_v, labels_val)
preds_dn = (combo_t >= thr_dn).astype(int)

f1_b2 = f1_score(labels_test, preds_b2, average='macro', zero_division=0)
f1_dn = f1_score(labels_test, preds_dn, average='macro', zero_division=0)
print(f"B2 macro-F1 (sanity): {f1_b2:.4f}")
print(f"doc_neighbours macro-F1 (sanity): {f1_dn:.4f}")
print(f"Lift: {f1_dn - f1_b2:+.4f}")

# 1. Flip-bucket analysis
correct_b2 = (preds_b2 == labels_test).astype(int)
correct_dn = (preds_dn == labels_test).astype(int)
bucket_RR = (correct_b2 == 1) & (correct_dn == 1)
bucket_RW = (correct_b2 == 1) & (correct_dn == 0)
bucket_WR = (correct_b2 == 0) & (correct_dn == 1)
bucket_WW = (correct_b2 == 0) & (correct_dn == 0)

n_total = labels_test.size
print(f"\n=== Flip buckets on (paper, goal) pairs (n={n_total} = 10000 x 17) ===")
counts = [
    ('RR (both right)',          int(bucket_RR.sum())),
    ('WR (neighbours fix B2)',   int(bucket_WR.sum())),
    ('RW (neighbours break B2)', int(bucket_RW.sum())),
    ('WW (both wrong)',          int(bucket_WW.sum())),
]
for name, cnt in counts:
    print(f"  {name:35s}  {cnt:>7,}  ({100*cnt/n_total:5.1f}%)")
print(f"  Net flips in favour of neighbours (WR - RW): "
      f"{counts[1][1] - counts[2][1]:+,}")

# 2. Per-paper features
sims_test = test_emb @ pool_emb.T
nn_idx = np.argsort(-sims_test, axis=1)[:, :K_best]
nn_sims = np.take_along_axis(sims_test, nn_idx, axis=1)
mean_cosine = nn_sims.mean(axis=1)

n_nan = int(np.isnan(pool_labels_goal).sum())
if n_nan > 0:
    print(f"\n[INFO] pool_labels_goal had {n_nan} NaN values — replacing with 0 for entropy.")
pool_labels_clean = np.nan_to_num(pool_labels_goal, nan=0.0)
nn_labels = pool_labels_clean[nn_idx]
goal_prob_in_neighbourhood = nn_labels.mean(axis=1)

def goal_prob_to_entropy(p):
    p = np.clip(p, 1e-9, 1 - 1e-9).astype(np.float64)
    H = -(p * np.log2(p) + (1 - p) * np.log2(1 - p))
    H = np.where(np.isfinite(H), H, 0.0)
    return H.mean(axis=-1)

nbr_entropy = goal_prob_to_entropy(goal_prob_in_neighbourhood)
print(f"nbr_entropy: range [{nbr_entropy.min():.4f}, {nbr_entropy.max():.4f}], "
      f"NaN count: {int(np.isnan(nbr_entropy).sum())}")

def categorical_entropy(probs):
    p = probs / np.clip(probs.sum(axis=-1, keepdims=True), 1e-9, None)
    return -np.sum(p * np.log2(np.clip(p, 1e-9, 1)), axis=-1)
llm_entropy = categorical_entropy(g_t)
llm_max = g_t.max(axis=1)

# 3. Per-feature comparison
per_paper_WR = bucket_WR.sum(axis=1)
per_paper_RW = bucket_RW.sum(axis=1)
per_paper_net = per_paper_WR - per_paper_RW

print('\n=== Which paper-level features predict the gain? ===')
print(f'{"Feature":40s}  {"low-half WR":>12s} {"high-half WR":>13s}  {"low-half net":>12s} {"high-half net":>14s}  {"r(feat,net)":>13s}')
print('-' * 115)

for name, feat in [
    ('top-K mean cosine (reliability)',     mean_cosine),
    ('top-K label consensus (-entropy)',    -nbr_entropy),
    ('LLM goal-prob entropy (uncertainty)', llm_entropy),
    ('LLM uncertainty (-max prob)',         -llm_max),
]:
    median = np.median(feat)
    low_mask  = feat <= median
    high_mask = feat >  median
    low_wr  = per_paper_WR[low_mask].mean()
    high_wr = per_paper_WR[high_mask].mean()
    low_net  = per_paper_net[low_mask].mean()
    high_net = per_paper_net[high_mask].mean()
    r = np.corrcoef(feat, per_paper_net)[0, 1]
    print(f'{name:40s}  {low_wr:>12.3f} {high_wr:>13.3f}  {low_net:>+12.3f} {high_net:>+14.3f}  {r:>+13.3f}')

# 4. Cross-tab by LLM uncertainty x neighbour reliability (cosine)
print('\n=== Cross-tabulation: gain concentration by LLM uncertainty x neighbour reliability ===')
llm_unc_med = np.median(llm_entropy)
nbr_rel_med = np.median(mean_cosine)
llm_uncertain = llm_entropy >  llm_unc_med
llm_certain   = llm_entropy <= llm_unc_med
nbr_reliable  = mean_cosine >  nbr_rel_med
nbr_unreliable = mean_cosine <= nbr_rel_med

for name, mask in [
    ('LLM uncertain + neighbours reliable',   llm_uncertain & nbr_reliable),
    ('LLM uncertain + neighbours unreliable', llm_uncertain & nbr_unreliable),
    ('LLM certain + neighbours reliable',     llm_certain   & nbr_reliable),
    ('LLM certain + neighbours unreliable',   llm_certain   & nbr_unreliable),
]:
    n_papers = int(mask.sum())
    mean_wr = per_paper_WR[mask].mean() if n_papers else 0
    mean_rw = per_paper_RW[mask].mean() if n_papers else 0
    mean_net = per_paper_net[mask].mean() if n_papers else 0
    print(f'  {name:42s}  n={n_papers:>5}  '
          f'avg WR={mean_wr:>5.3f}  avg RW={mean_rw:>5.3f}  '
          f'avg net={mean_net:>+5.3f}')

# 4b. Cross-tab by LLM uncertainty x label-CONSENSUS (the v2-fixed feature)
print('\n=== Cross-tabulation: gain by LLM uncertainty x neighbour-label CONSENSUS ===')
nbr_cons_med = np.median(-nbr_entropy)
nbr_consensus    = (-nbr_entropy) >  nbr_cons_med
nbr_no_consensus = (-nbr_entropy) <= nbr_cons_med

for name, mask in [
    ('LLM uncertain + neighbour consensus',     llm_uncertain & nbr_consensus),
    ('LLM uncertain + neighbour disagreement',  llm_uncertain & nbr_no_consensus),
    ('LLM certain + neighbour consensus',       llm_certain   & nbr_consensus),
    ('LLM certain + neighbour disagreement',    llm_certain   & nbr_no_consensus),
]:
    n_papers = int(mask.sum())
    mean_wr = per_paper_WR[mask].mean() if n_papers else 0
    mean_rw = per_paper_RW[mask].mean() if n_papers else 0
    mean_net = per_paper_net[mask].mean() if n_papers else 0
    print(f'  {name:42s}  n={n_papers:>5}  '
          f'avg WR={mean_wr:>5.3f}  avg RW={mean_rw:>5.3f}  '
          f'avg net={mean_net:>+5.3f}')

# 5. Save
import os, pandas as pd
out_dir = os.path.join(DRIVE_ROOT, 'cross_model_comparison_v5')
df_papers = pd.DataFrame({
    'mean_cosine':   mean_cosine,
    'nbr_entropy':   nbr_entropy,
    'llm_entropy':   llm_entropy,
    'llm_max':       llm_max,
    'per_paper_WR':  per_paper_WR,
    'per_paper_RW':  per_paper_RW,
    'per_paper_net': per_paper_net,
})
df_papers.to_csv(os.path.join(out_dir, 'flip_analysis_per_paper.csv'), index=False)
print(f'\nSaved per-paper data: {out_dir}/flip_analysis_per_paper.csv')


In [ ]:
print(f'pool_labels_goal NaN count: {np.isnan(pool_labels_goal).sum()}')
print(f'goal_prob_in_neighbourhood NaN count: {np.isnan(goal_prob_in_neighbourhood).sum()}')
print(f'nbr_entropy NaN count: {np.isnan(nbr_entropy).sum()}')
print(f'nbr_entropy range: [{np.nanmin(nbr_entropy):.4f}, {np.nanmax(nbr_entropy):.4f}]')

In [ ]:
# Export split manifests for release. CPU only; no LLM, no network.
# Paths derive from SDG_ROOT via the resolver in the setup cell.

import os, re, json, hashlib, numpy as np, pandas as pd

SPLIT_DIR = os.path.join(SDG_ROOT, 'artifacts', 'splits')
os.makedirs(SPLIT_DIR, exist_ok=True)

def _sha256(s):
    return hashlib.sha256(str(s).encode('utf-8')).hexdigest()

def export_split(df, name):
    gcols = sorted([c for c in df.columns if re.match(r'^sdg_\d+$', c)],
                   key=lambda c: int(c.split('_')[1]))
    tcols = sorted([c for c in df.columns if c.startswith('target_')])

    pd.DataFrame({
        'doi':             df['doi'].astype(str).values,
        'title':           df['title'].astype(str).values,
        'abstract_sha256': [_sha256(a) for a in df['abstract'].astype(str).values],
        'abstract_nchar':  df['abstract'].astype(str).str.len().values,
    }).to_csv(os.path.join(SPLIT_DIR, f'{name}_manifest.csv'), index=False)

    with open(os.path.join(SPLIT_DIR, f'{name}_dois.txt'), 'w') as f:
        f.write('\n'.join(df['doi'].astype(str).tolist()) + '\n')

    np.savez_compressed(
        os.path.join(SPLIT_DIR, f'{name}_labels.npz'),
        goal=df[gcols].values.astype(np.int8),
        target=(df[tcols].values.astype(np.int8) if tcols
                else np.zeros((len(df), 0), np.int8)),
        goal_cols=np.array(gcols),
        target_cols=np.array(tcols))

    fp = _hash_dois(df)
    print(f'  {name:8s} n={len(df):>6}  goals={len(gcols):>3} '
          f'targets={len(tcols):>4}  fingerprint={fp}')
    return {'split': name, 'n': int(len(df)), 'fingerprint': fp,
            'n_goal_cols': len(gcols), 'n_target_cols': len(tcols)}

meta = [export_split(d, n) for d, n in
        [(df_fewshot, 'fewshot'), (df_val, 'val'), (df_test, 'test')]]

got = {m['split']: m['fingerprint'] for m in meta}
assert got['test'] == ORIGINAL_HASHES['test'] and got['val'] == ORIGINAL_HASHES['val'], (
    f"FINGERPRINT MISMATCH: test={got['test']} val={got['val']}, "
    f"expected {ORIGINAL_HASHES}. This parquet is not the one behind the paper. "
    "Do not release.")
assert len(df_test) == N_TEST, f'test split is {len(df_test)} rows, expected {N_TEST}'
print('  fingerprints OK - this is the evaluated split')

try:
    meta.append(export_split(df_pool, 'pool'))
except NameError:
    print('  WARNING: df_pool undefined - run cell 11 and re-run this cell.')

with open(os.path.join(SPLIT_DIR, 'splits_meta.json'), 'w') as f:
    json.dump({'seed': SEED,
               'openalex_fetched': '2026-05-05',
               'aurora_label_doi': '10.5281/zenodo.5224005',
               'stratum': '2-5 assigned goals',
               'splits': meta}, f, indent=2)

print(f'\nWrote {SPLIT_DIR}')

In [ ]:
# Verifies the introduction's claim about the FULL Aurora corpus.
# Run in the Compare/Ablate notebook after cell 3 (needs DATA_CACHE only).
# CPU, no LLM, ~30 seconds.

import os, re, numpy as np, pandas as pd

p = os.path.join(DATA_CACHE, 'aurora_multilabel_full.parquet')
assert os.path.exists(p), f'missing {p}'
df = pd.read_parquet(p, columns=None)

gcols = sorted([c for c in df.columns if re.match(r'^sdg_\d+$', c)],
               key=lambda c: int(c.split('_')[1]))
assert len(gcols) == 17, f'found {len(gcols)} goal columns'

G   = df[gcols].values.astype(np.int8)
pos = G.mean(0) * 100
names = ["No Poverty","Zero Hunger","Good Health","Quality Education","Gender Equality",
         "Clean Water","Clean Energy","Decent Work","Innovation","Reduced Inequalities",
         "Sustainable Cities","Responsible Consumption","Climate Action","Life Below Water",
         "Life on Land","Peace & Justice","Partnerships"]

print(f'FULL Aurora corpus: {len(df):,} papers\n')
for i in np.argsort(-pos):
    print(f'  SDG {i+1:>2}  {names[i]:<24} {pos[i]:>6.2f}%')

print(f'\n  most common : SDG {pos.argmax()+1} at {pos.max():.2f}%')
print(f'  least common: SDG {pos.argmin()+1} at {pos.min():.2f}%')
print(f'  max/min ratio = {pos.max()/pos.min():.1f}x')
print(f'\n  paper intro claims: goals 3, 9, 13 dominate; 14 and 17 an order of magnitude fewer')
rank = list(np.argsort(-pos))
for g in (3, 9, 13, 14, 17):
    print(f'    SDG {g:>2}: {pos[g-1]:>6.2f}%   rank {rank.index(g-1)+1} of 17')
print(f'\n  -> "order of magnitude" (10x) supported: {pos.max()/pos.min() >= 10}')

In [ ]:
# Verifies Table 5's three data columns for Gemma. CPU only, no GPU, no LLM.
# Paste into Aurora_SDG_MultiModel_Compare_Ablate_v5.ipynb AFTER cell 15.
# Runs in about a minute. Copy the printed block back to me.

import numpy as np
from sklearn.metrics import f1_score

m = MODELS[0]          # 'gemma4-26b'
K = 50

g_v = cached[m]['goal_val'];    g_t = cached[m]['goal_test']
t_v = cached[m]['target_val'];  t_t = cached[m]['target_test']

# --- B2: target-LLM aggregated, per goal ---
gva = aggregate_targets_to_goals(t_v)
gta = aggregate_targets_to_goals(t_t)
thr_b2  = opt_threshold_per_class(gva, labels_val)
pred_b2 = (gta >= thr_b2).astype(int)

# --- doc_neighbours alone, K=50, global lambda (the paper's configuration) ---
evid_v = doc_neighbours_evidence(val_emb,  pool_emb, pool_labels_goal, K=K)
evid_t = doc_neighbours_evidence(test_emb, pool_emb, pool_labels_goal, K=K)
lam, _ = calibrate_lambda_global(g_v, evid_v, labels_val.astype(np.float32))
combo_v = (1 - lam) * g_v + lam * evid_v
combo_t = (1 - lam) * g_t + lam * evid_t
thr_nbr  = opt_threshold_per_class(combo_v, labels_val)
pred_nbr = (combo_t >= thr_nbr).astype(int)

names = ["No Poverty","Zero Hunger","Good Health","Quality Education","Gender Equality",
         "Clean Water","Clean Energy","Decent Work","Innovation","Reduced Inequalities",
         "Sustainable Cities","Responsible Consumption","Climate Action","Life Below Water",
         "Life on Land","Peace & Justice","Partnerships"]

print(f'global lambda = {lam:.2f}   K = {K}')
print(f'{"SDG":>4} {"name":<24}{"B2 F1":>9}{"doc_nbr F1":>12}{"delta (pp)":>12}')
print('-' * 62)
b2s, nbrs = [], []
for g in range(17):
    f_b2  = f1_score(labels_test[:, g], pred_b2[:, g],  zero_division=0)
    f_nbr = f1_score(labels_test[:, g], pred_nbr[:, g], zero_division=0)
    b2s.append(f_b2); nbrs.append(f_nbr)
    print(f'{g+1:>4} {names[g]:<24}{f_b2:>9.4f}{f_nbr:>12.4f}{(f_nbr-f_b2)*100:>+12.2f}')
print('-' * 62)
print(f'{"":>4} {"macro":<24}{np.mean(b2s):>9.4f}{np.mean(nbrs):>12.4f}'
      f'{(np.mean(nbrs)-np.mean(b2s))*100:>+12.2f}')
print('\nCHECK: macro doc_nbr should read 0.6088 and macro B2 0.5359.')

In [ ]:
# =============================================================================
# CELL U - Unified evaluation: Tables 3-6 under ONE threshold routine,
#          + Gaussian over 20 seeds, + bootstrap CIs, + retrieval-only baseline.
#
# CPU only. No LLM. No GPU. Nothing is overwritten: output goes to a new
# folder, cross_model_comparison_v6/.
#
# It checks all its inputs FIRST and stops immediately with a clear message if
# something is missing, so you never lose ten minutes to a late crash.
# =============================================================================

import os, json, itertools
import numpy as np, pandas as pd
from sklearn.metrics import f1_score

# ---- CONFIG -----------------------------------------------------------------
# The canonical threshold grid. The paper (Sec 6.2) states {0.10,...,0.85}.
# Cell 13 actually uses np.arange(0.05,0.96,0.05). We score EVERYTHING through
# one grid; set which one here. Both are reported for B1/B2 in Step 1 so you
# can see the size of the discrepancy before choosing.
CANON_GRID   = np.arange(0.10, 0.851, 0.05)     # paper's stated grid, 16 points
LEGACY_GRID  = np.arange(0.05, 0.96,  0.05)     # Cell 13's grid, 19 points

ALPHAS    = [0.70, 0.80, 0.85, 0.90, 0.95]
CONFS     = [0.50, 0.60, 0.70]
WB_GRID   = [0.0, 0.25, 0.50, 0.75, 1.0]
N_SEEDS_G = 20
N_BOOT    = 1000
K_HEAD    = 50
BOOT_SEED = 12345

# ---- PREFLIGHT: verify every input exists before doing any work -------------
_need = ['cached', 'MODELS', 'labels_val', 'labels_test', 'GOAL_TO_TARGETS',
         'aggregate_targets_to_goals', 'pool_emb', 'pool_labels_goal',
         'val_emb', 'test_emb', 'DATA_CACHE', 'DRIVE_ROOT']
_g = globals()
_missing = [n for n in _need if n not in _g]
if _missing:
    print('MISSING: ' + ', '.join(_missing))
    print('\nPresent from the list I need: ' +
          (', '.join(n for n in _need if n in _g) or '(none)'))
    print('\nIf ALL are missing, the kernel has no notebook state - run the '
          'notebook top to bottom, then re-run this cell.')
    print('If only some are missing, the names differ in your version. '
          'Candidates currently defined:')
    for _k, _v in sorted(_g.items()):
        if _k.startswith('_'):
            continue
        if isinstance(_v, np.ndarray):
            print(f'    {_k:28s} ndarray {_v.shape}')
        elif isinstance(_v, dict) and _v and all(isinstance(x, dict) for x in _v.values()):
            print(f'    {_k:28s} dict of dicts, keys={list(_v)[:5]}')
        elif isinstance(_v, (list, tuple)) and 0 < len(_v) <= 6:
            print(f'    {_k:28s} {type(_v).__name__} {_v}')
        elif isinstance(_v, str) and ('/' in _v):
            print(f'    {_k:28s} path {_v}')
    raise RuntimeError('Preflight failed - see the list above.')

print('PREFLIGHT')
print(f'  MODELS ({len(MODELS)}): {MODELS}')
if len(MODELS) < 4:
    print('  !! Fewer than 4 backbones. The cross-backbone claims need all four.')
for _m in MODELS:
    _c = cached[_m]
    print(f'    {_m:15s} goal_test={_c["goal_test"].shape} target_test={_c["target_test"].shape}')
print(f'  labels_val={np.asarray(labels_val).shape}  labels_test={np.asarray(labels_test).shape}')
print(f'  pool_emb={pool_emb.shape}  pool_labels_goal={pool_labels_goal.shape}')
print(f'  val_emb={val_emb.shape}  test_emb={test_emb.shape}')

for _m in MODELS:
    assert cached[_m]['goal_test'].shape[0] == np.asarray(labels_test).shape[0], \
        f'{_m}: goal_test rows != labels_test rows'
    assert cached[_m]['goal_val'].shape[0] == np.asarray(labels_val).shape[0], \
        f'{_m}: goal_val rows != labels_val rows'
assert val_emb.shape[0] == np.asarray(labels_val).shape[0], 'val_emb rows != labels_val'
assert test_emb.shape[0] == np.asarray(labels_test).shape[0], 'test_emb rows != labels_test'
print('  shapes consistent.\n')

OUT = os.path.join(DRIVE_ROOT, 'cross_model_comparison_v6')
os.makedirs(OUT, exist_ok=True)
print(f'Output dir: {OUT}\n')

# ---- CANONICAL SCORING (single implementation, used by every row below) ------
def fit_thresholds(probs, labels, grid=CANON_GRID):
    C = probs.shape[1]
    thr = np.zeros(C, dtype=np.float64)
    for c in range(C):
        best_f, best_t = -1.0, float(grid[0])
        yc = labels[:, c]
        for t in grid:
            f = f1_score(yc, (probs[:, c] >= t).astype(int), zero_division=0)
            if f > best_f:
                best_f, best_t = f, float(t)
        thr[c] = best_t
    return thr

def score(probs, labels, thr):
    return f1_score(labels, (probs >= thr).astype(int),
                    average='macro', zero_division=0)

def fit_and_score(p_val, y_val, p_test, y_test, grid=CANON_GRID):
    """Fit thresholds on val, apply to test. Returns (val_f1, test_f1, thr)."""
    thr = fit_thresholds(p_val, y_val, grid)
    return score(p_val, y_val, thr), score(p_test, y_test, thr), thr

y_val  = np.asarray(labels_val).astype(int)
y_test = np.asarray(labels_test).astype(int)

# =============================================================================
# STEP 1 — Quantify the two-routine discrepancy (closes F02's diagnosis)
# =============================================================================
print('='*78); print('STEP 1: threshold-grid sensitivity of the baselines'); print('='*78)
grid_rows = []
for m in MODELS:
    gv, gt = cached[m]['goal_val'],   cached[m]['goal_test']
    av     = aggregate_targets_to_goals(cached[m]['target_val'])
    at     = aggregate_targets_to_goals(cached[m]['target_test'])
    for gname, grid in [('paper_0.10-0.85', CANON_GRID), ('cell13_0.05-0.95', LEGACY_GRID)]:
        _, b1, _ = fit_and_score(gv, y_val, gt, y_test, grid)
        _, b2, _ = fit_and_score(av, y_val, at, y_test, grid)
        grid_rows.append({'model': m, 'grid': gname, 'B1': b1, 'B2': b2})
df_grid = pd.DataFrame(grid_rows)
print(df_grid.pivot(index='model', columns='grid', values=['B1','B2']).round(4).to_string())
df_grid.to_csv(os.path.join(OUT, 'threshold_grid_sensitivity.csv'), index=False)
print('\n-> If the gap here matches 0.5216 vs 0.5223 (B1) and 0.5286 vs 0.5359 (B2),')
print('   the grid IS the cause and Sec 6.4 must be rewritten.\n')

# =============================================================================
# STEP 2 — Build all seven matrices in THIS notebook
# =============================================================================
print('='*78); print('STEP 2: building matrices A-G'); print('='*78)
goal_of = {i: g for g in range(1, 18) for i in GOAL_TO_TARGETS[g]}

# --- A: Pradhan goal-level, from the released JSON ---
def _find(fname, roots):
    """Search a few roots recursively for a filename. Returns first hit or None."""
    import glob
    for r in roots:
        if not r or not os.path.isdir(r):
            continue
        hits = glob.glob(os.path.join(r, '**', fname), recursive=True)
        if hits:
            return sorted(hits, key=len)[0]
    return None

# Where to look for the released matrices. All entries are derived from the
# notebook's own path variables; nothing is hard-coded. If your checkout keeps
# the interlinkage matrix outside the project tree, add its directory here.
EXTRA_MATRIX_DIRS = []
_ROOTS = [d for d in ([DATA_CACHE, DRIVE_ROOT,
                       os.path.dirname(DRIVE_ROOT.rstrip('/'))]
                      + list(EXTRA_MATRIX_DIRS)) if d]

W_A = None
_pj_path = _find('sdg_interaction_matrix_v3_2.json', _ROOTS) or \
           _find('sdg_interaction_matrix*.json', _ROOTS)
if _pj_path:
    with open(_pj_path) as f:
        pj = json.load(f)
    _key = 'matrix_W' if 'matrix_W' in pj else next(
        (k for k, v in pj.items() if isinstance(v, list) and len(v) == 17), None)
    if _key:
        W_A = np.array(pj[_key], dtype=np.float64); np.fill_diagonal(W_A, 0.0)
        print(f'  A  loaded from {_pj_path}  (key "{_key}")')
    else:
        print(f'  A  found {_pj_path} but no 17x17 array inside; keys={list(pj)[:8]}')
if W_A is None:
    print('  A  MISSING - Pradhan JSON not found; A and B will be skipped.')

# --- B: inherit A to target level ---
W_B = None
if W_A is not None:
    gi = np.array([goal_of[i] - 1 for i in range(169)])
    W_B = W_A[np.ix_(gi, gi)].copy()
    W_B[gi[:, None] == gi[None, :]] = 0.0
    print(f'  B  built, nnz={int((W_B!=0).sum())}')

# --- C: SBERT on target descriptors ---
W_C = None
p_c = _find('sbert_target_matrix_169.npy', _ROOTS)
if p_c:
    W_C = np.load(p_c).astype(np.float64); np.fill_diagonal(W_C, 0.0)
    print(f'  C  loaded from {p_c}')
else:
    try:
        desc = [TARGET_DESC[t] for t in TARGET_IDS]
        E = sbert.encode(desc, normalize_embeddings=True, batch_size=64,
                         show_progress_bar=False).astype(np.float64)
        W_C = E @ E.T; np.fill_diagonal(W_C, 0.0)
        print('  C  re-encoded with SBERT')
    except Exception as e:
        print(f'  C  MISSING ({e})')

def center_rescale(W, ref):
    """Median-centre off-diagonal, rescale abs-extremum to match ref."""
    W = W.astype(np.float64).copy(); np.fill_diagonal(W, 0.0)
    iu = np.triu_indices(W.shape[0], 1)
    med = np.median(W[iu]); W = W - med; np.fill_diagonal(W, 0.0)
    if ref is not None and np.abs(W).max() > 0:
        W = W * (np.abs(ref).max() / np.abs(W).max())
    return W

if W_C is not None:
    W_C = center_rescale(W_C, W_B)

# --- E: UN 5P taxonomy ---
FIVE_P = {1:'People',2:'People',3:'People',4:'People',5:'People',
          6:'Planet',12:'Planet',13:'Planet',14:'Planet',15:'Planet',
          7:'Prosperity',8:'Prosperity',9:'Prosperity',10:'Prosperity',11:'Prosperity',
          16:'Peace',17:'Partnership'}
W_E = np.full((169,169), 0.05)
for i in range(169):
    for j in range(169):
        if i == j: continue
        gi_, gj_ = goal_of[i], goal_of[j]
        if gi_ == gj_:                       W_E[i,j] = 0.35
        elif FIVE_P[gi_] == FIVE_P[gj_]:     W_E[i,j] = 0.20
np.fill_diagonal(W_E, 0.0)
print('  E  built (three-valued, NOT centred - nonnegative by construction)')

# --- F: keyword Jaccard ---
W_F = None
p_f = _find('keyword_matrix_169*.npy', _ROOTS)
if p_f:
    W_F = center_rescale(np.load(p_f), W_B); print(f'  F  loaded from {p_f}')
else:
    try:
        import re
        STOP = set('''a an the and or of to in for on with by from at as is are be been being
        this that these those it its their his her they them we our you your not no all any
        such other more most than then so if into over under between within through during'''.split())
        Ks = []
        for t in TARGET_IDS:
            toks = re.findall(r'[a-z]+', TARGET_DESC[t].lower())
            Ks.append({w for w in toks if len(w) >= 3 and w not in STOP})
        W = np.zeros((169,169))
        for i in range(169):
            for j in range(i+1, 169):
                u = len(Ks[i] | Ks[j])
                v = len(Ks[i] & Ks[j]) / u if u else 0.0
                W[i,j] = W[j,i] = v
        W_F = center_rescale(W, W_B); print('  F  built from target descriptors')
    except Exception as e:
        print(f'  F  MISSING ({e})')

# --- G: Gaussian, seeded ---
def make_G(seed, ref):
    rs = np.random.RandomState(seed)
    W = rs.normal(0.0, ref.std(), size=(169,169))
    W = (W + W.T) / np.sqrt(2.0)      # symmetrise WITHOUT halving variance
    np.fill_diagonal(W, 0.0)
    return W

# =============================================================================
# STEP 3 — Propagation operator (identical to Cell 37)
# =============================================================================
def propagate(probs, W, alpha, conf):
    Wp = np.where(W > 0, W, 0.0).copy()
    Wn = np.abs(np.where(W < 0, W, 0.0)).copy()
    rp = Wp.sum(1, keepdims=True); rp[rp == 0] = 1.0; Wp /= rp
    rn = Wn.sum(1, keepdims=True); rn[rn == 0] = 1.0; Wn /= rn
    gated = probs * (probs >= conf)
    return np.clip(alpha*probs + (1-alpha)*(gated @ Wp.T) - (1-alpha)*(gated @ Wn.T), 0, 1)

def eval_graph(m, W, level, alpha, conf):
    """level 'goal' -> 17x17 on goal probs; 'target' -> 169x169 then aggregate."""
    if level == 'goal':
        pv = propagate(cached[m]['goal_val'].astype(np.float64),  W, alpha, conf)
        pt = propagate(cached[m]['goal_test'].astype(np.float64), W, alpha, conf)
        return fit_and_score(pv, y_val, pt, y_test) + (None, None)
    tv = propagate(cached[m]['target_val'].astype(np.float64),  W, alpha, conf)
    tt = propagate(cached[m]['target_test'].astype(np.float64), W, alpha, conf)
    v, t, _ = fit_and_score(aggregate_targets_to_goals(tv), y_val,
                            aggregate_targets_to_goals(tt), y_test)
    return v, t, None, tv, tt

# =============================================================================
# STEP 4 — Table 3 regenerated under the canonical routine
# =============================================================================
print('\n'+'='*78); print('STEP 4: graph table (val-selected, canonical routine)'); print('='*78)
MATS = [('A', W_A, 'goal'), ('B', W_B, 'target'), ('C', W_C, 'target'),
        ('E', W_E, 'target'), ('F', W_F, 'target'),
        ('G', (make_G(42, W_B) if W_B is not None else None), 'target')]

rows = []
D_BY_MODEL = {}
for m in MODELS:
    gv, gt = cached[m]['goal_val'], cached[m]['goal_test']
    av = aggregate_targets_to_goals(cached[m]['target_val'])
    at = aggregate_targets_to_goals(cached[m]['target_test'])
    b1v, b1, _ = fit_and_score(gv, y_val, gt, y_test)
    b2v, b2, _ = fit_and_score(av, y_val, at, y_test)
    rows += [{'model':m,'matrix':'B1','alpha':None,'conf':None,'w_B':None,'val_f1':b1v,'test_f1':b1},
             {'model':m,'matrix':'B2','alpha':None,'conf':None,'w_B':None,'val_f1':b2v,'test_f1':b2}]
    for name, W, level in MATS:
        if W is None:
            print(f'  {m:15s} {name}: skipped (matrix unavailable)'); continue
        best = None
        for a, c in itertools.product(ALPHAS, CONFS):
            v, t, _, _, _ = eval_graph(m, W, level, a, c)
            if best is None or v > best[0]: best = (v, t, a, c)
        rows.append({'model':m,'matrix':name,'alpha':best[2],'conf':best[3],
                     'w_B':None,'val_f1':best[0],'test_f1':best[1]})
        print(f'  {m:15s} {name}: val={best[0]:.4f} test={best[1]:.4f} (a={best[2]}, conf={best[3]})')
    # D: joint search over w_B
    if W_B is not None and W_C is not None:
        best = None
        for wb in WB_GRID:
            W_D = wb*W_B + (1-wb)*W_C
            for a, c in itertools.product(ALPHAS, CONFS):
                v, t, _, _, _ = eval_graph(m, W_D, 'target', a, c)
                if best is None or v > best[0]: best = (v, t, a, c, wb)
        rows.append({'model':m,'matrix':'D','alpha':best[2],'conf':best[3],
                     'w_B':best[4],'val_f1':best[0],'test_f1':best[1]})
        D_BY_MODEL[m] = best[4]*W_B + (1-best[4])*W_C
        print(f'  {m:15s} D: val={best[0]:.4f} test={best[1]:.4f} (a={best[2]}, conf={best[3]}, w_B={best[4]})')

df_graph = pd.DataFrame(rows)
df_graph.to_csv(os.path.join(OUT, 'table3_unified.csv'), index=False)
print('\n' + df_graph.pivot_table(index='matrix', columns='model',
                                  values='test_f1').round(4).to_string())

# =============================================================================
# STEP 5 — Gaussian control over 20 seeds (F19)
# =============================================================================
print('\n'+'='*78); print(f'STEP 5: Gaussian control, {N_SEEDS_G} seeds'); print('='*78)
g_rows = []
if W_B is not None:
    for m in MODELS:
        vals = []
        for s in range(N_SEEDS_G):
            W = make_G(1000 + s, W_B)
            best = None
            for a, c in itertools.product(ALPHAS, CONFS):
                v, t, _, _, _ = eval_graph(m, W, 'target', a, c)
                if best is None or v > best[0]: best = (v, t)
            vals.append(best[1]); g_rows.append({'model':m,'seed':1000+s,'test_f1':best[1]})
        vals = np.array(vals)
        print(f'  {m:15s} G over {N_SEEDS_G} seeds: mean={vals.mean():.4f} '
              f'sd={vals.std(ddof=1):.4f} min={vals.min():.4f} max={vals.max():.4f}')
    pd.DataFrame(g_rows).to_csv(os.path.join(OUT, 'gaussian_seeds.csv'), index=False)

# =============================================================================
# STEP 6 — Retrieval-only baseline, lambda = 1 (F07)
# =============================================================================
print('\n'+'='*78); print('STEP 6: retrieval-only (lambda=1) and neighbour mixtures'); print('='*78)

def nbr_vote(q_emb, p_emb, p_lab, K, weighted=True):
    """weighted=True implements Eq.5 (similarity-weighted). False = Cell 13's mean."""
    sims = q_emb @ p_emb.T
    idx  = np.argpartition(-sims, K-1, axis=1)[:, :K]
    out  = np.zeros((q_emb.shape[0], p_lab.shape[1]), dtype=np.float64)
    for i in range(q_emb.shape[0]):
        nn = idx[i]
        if weighted:
            w = np.clip(sims[i, nn], 0, None); Z = w.sum()
            out[i] = (w @ p_lab[nn]) / Z if Z > 0 else p_lab[nn].mean(0)
        else:
            out[i] = p_lab[nn].mean(0)
    return out

doc_rows = []
for m in MODELS:
    gv, gt = cached[m]['goal_val'].astype(np.float64), cached[m]['goal_test'].astype(np.float64)
    for wtd in [True, False]:
        ev = nbr_vote(val_emb,  pool_emb, pool_labels_goal, K_HEAD, wtd)
        et = nbr_vote(test_emb, pool_emb, pool_labels_goal, K_HEAD, wtd)
        # retrieval only
        rv, rt, _ = fit_and_score(ev, y_val, et, y_test)
        # calibrated mixture
        best = None
        for lam in np.arange(0.0, 1.01, 0.1):
            cv_, ct_ = (1-lam)*gv + lam*ev, (1-lam)*gt + lam*et
            v, t, _ = fit_and_score(cv_, y_val, ct_, y_test)
            if best is None or v > best[0]: best = (v, t, lam)
        doc_rows.append({'model':m, 'vote':'weighted_Eq5' if wtd else 'unweighted_cell13',
                         'retrieval_only_val':rv, 'retrieval_only_test':rt,
                         'mix_val':best[0], 'mix_test':best[1], 'lambda':best[2]})
        print(f'  {m:15s} {"Eq5-weighted" if wtd else "cell13-mean ":14s} '
              f'lam=1 test={rt:.4f} | mix lam={best[2]:.1f} test={best[1]:.4f}')
df_doc = pd.DataFrame(doc_rows)
df_doc.to_csv(os.path.join(OUT, 'retrieval_only_and_mixtures.csv'), index=False)

# =============================================================================
# STEP 7 — Document-level bootstrap CIs (F12)
# =============================================================================
print('\n'+'='*78); print(f'STEP 7: paired bootstrap, {N_BOOT} resamples'); print('='*78)

def boot_ci(pA, pB, thrA, thrB, n_boot=N_BOOT, seed=BOOT_SEED):
    """Paired macro-F1 difference (A - B), resampling documents."""
    rs = np.random.RandomState(seed); n = pA.shape[0]; d = []
    for _ in range(n_boot):
        b = rs.randint(0, n, n)
        d.append(score(pA[b], y_test[b], thrA) - score(pB[b], y_test[b], thrB))
    d = np.array(d)
    return d.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5)

ci_rows = []
for m in MODELS:
    gv, gt = cached[m]['goal_val'].astype(np.float64), cached[m]['goal_test'].astype(np.float64)
    av = aggregate_targets_to_goals(cached[m]['target_val'])
    at = aggregate_targets_to_goals(cached[m]['target_test'])
    thr_b1 = fit_thresholds(gv, y_val); thr_b2 = fit_thresholds(av, y_val)

    ev = nbr_vote(val_emb, pool_emb, pool_labels_goal, K_HEAD, True)
    et = nbr_vote(test_emb, pool_emb, pool_labels_goal, K_HEAD, True)
    lam = float(df_doc[(df_doc.model==m)&(df_doc.vote=='weighted_Eq5')]['lambda'].iloc[0])
    nv, nt = (1-lam)*gv + lam*ev, (1-lam)*gt + lam*et
    thr_n = fit_thresholds(nv, y_val)

    for lbl, pA, tA, pB, tB in [('nbr - B1', nt, thr_n, gt, thr_b1),
                                 ('nbr - B2', nt, thr_n, at, thr_b2)]:
        mu, lo, hi = boot_ci(pA, pB, tA, tB)
        ci_rows.append({'model':m,'comparison':lbl,'mean_diff':mu,'ci_lo':lo,'ci_hi':hi})
        print(f'  {m:15s} {lbl}: {mu:+.4f}  [{lo:+.4f}, {hi:+.4f}]')

    sel = df_graph[(df_graph.model==m) & (~df_graph.matrix.isin(['B1','B2']))]
    if len(sel):
        gname = sel.sort_values('val_f1', ascending=False).iloc[0]['matrix']
        r = sel[sel.matrix==gname].iloc[0]
        if gname == 'D':
            W, lvl = D_BY_MODEL.get(m), 'target'
        else:
            W, lvl = dict([(n,(w,l)) for n,w,l in MATS]).get(gname, (None,None))
        if W is not None:
            _, _, _, tv_, tt_ = eval_graph(m, W, lvl, r['alpha'], r['conf'])
            if lvl == 'goal':
                pv_ = propagate(gv, W, r['alpha'], r['conf'])
                pt_ = propagate(gt, W, r['alpha'], r['conf'])
            else:
                pv_, pt_ = aggregate_targets_to_goals(tv_), aggregate_targets_to_goals(tt_)
            thr_g = fit_thresholds(pv_, y_val)
            mu, lo, hi = boot_ci(pt_, at, thr_g, thr_b2)
            ci_rows.append({'model':m,'comparison':f'graph {gname} - B2',
                            'mean_diff':mu,'ci_lo':lo,'ci_hi':hi})
            print(f'  {m:15s} graph {gname} - B2: {mu:+.4f}  [{lo:+.4f}, {hi:+.4f}]')

pd.DataFrame(ci_rows).to_csv(os.path.join(OUT, 'bootstrap_ci.csv'), index=False)

print('\n'+'='*78)
print('DONE. Files written to', OUT)
for f in ['threshold_grid_sensitivity.csv','table3_unified.csv','gaussian_seeds.csv',
          'retrieval_only_and_mixtures.csv','bootstrap_ci.csv']:
    print('  ', f)
print('='*78)

In [ ]:
# =============================================================================
# CELL V - Regenerate EVERY remaining analysis under the canonical routine.
#
# Covers: target-level results, per-goal breakdowns, the K x lambda ablation,
# doc_entities and three-way fusion, Diagnostics 1-3, the prevalence
# regression, and the Aurora co-occurrence graph.
#
# CPU only. No LLM. No GPU. Output goes to cross_model_comparison_v6/.
# Expect 15-40 minutes depending on how many backbones are loaded.
# =============================================================================

import os, json, itertools, re
import numpy as np, pandas as pd
from sklearn.metrics import f1_score

# ---- PREFLIGHT --------------------------------------------------------------
_need = ['fit_thresholds', 'score', 'fit_and_score', 'propagate', 'eval_graph',
         'nbr_vote', 'CANON_GRID', 'ALPHAS', 'CONFS', 'MATS', 'df_graph',
         'y_val', 'y_test', 'OUT', 'K_HEAD', 'cached', 'MODELS',
         'GOAL_TO_TARGETS', 'aggregate_targets_to_goals',
         'pool_emb', 'pool_labels_goal', 'val_emb', 'test_emb']
_g = globals()
_missing = [n for n in _need if n not in _g]
if _missing:
    raise RuntimeError(
        'Missing: ' + ', '.join(_missing) +
        '\nRun CELL U first in the same session, then re-run this cell.')

print('PREFLIGHT ok - reusing Cell U definitions')

# ---- FAST SCORING -----------------------------------------------------------
# sklearn's f1_score dominates the inner loops. These numpy equivalents are
# verified against it below and produce identical values, so the routine is
# unchanged - only the implementation is faster.
def _f1_bin(yt, yp):
    tp = float(np.count_nonzero(yt & yp))
    fp = float(np.count_nonzero((~yt) & yp))
    fn = float(np.count_nonzero(yt & (~yp)))
    d = 2*tp + fp + fn
    return 0.0 if d == 0 else 2*tp/d

def fit_thresholds_fast(probs, labels, grid=CANON_GRID):
    C = probs.shape[1]
    yb = labels.astype(bool)
    thr = np.zeros(C)
    for c in range(C):
        col, yc = probs[:, c], yb[:, c]
        best_f, best_t = -1.0, float(grid[0])
        for t in grid:
            f = _f1_bin(yc, col >= t)
            if f > best_f:
                best_f, best_t = f, float(t)
        thr[c] = best_t
    return thr

def score_fast(probs, labels, thr):
    yb = labels.astype(bool)
    pred = probs >= thr
    return float(np.mean([_f1_bin(yb[:, c], pred[:, c]) for c in range(probs.shape[1])]))

def fas(p_val, y_v, p_test, y_t, grid=CANON_GRID):
    thr = fit_thresholds_fast(p_val, y_v, grid)
    return score_fast(p_val, y_v, thr), score_fast(p_test, y_t, thr), thr

# verify against Cell U's routine before relying on it
_pv = cached[MODELS[0]]['goal_val'].astype(np.float64)
_pt = cached[MODELS[0]]['goal_test'].astype(np.float64)
_a = fit_and_score(_pv, y_val, _pt, y_test)
_b = fas(_pv, y_val, _pt, y_test)
assert abs(_a[0]-_b[0]) < 1e-12 and abs(_a[1]-_b[1]) < 1e-12 and np.allclose(_a[2], _b[2]), \
    'fast scorer disagrees with Cell U - aborting'
print('  fast scorer verified identical to Cell U\n')

goal_of = {i: g for g in range(1, 18) for i in GOAL_TO_TARGETS[g]}
GOAL_NAMES = [f'SDG{g}' for g in range(1, 18)]
PRIMARY = 'gemma4-26b' if 'gemma4-26b' in MODELS else MODELS[0]
print(f'  primary backbone: {PRIMARY}\n')

def _valbest(m):
    """(matrix name, alpha, conf) selected on validation for backbone m."""
    sel = df_graph[(df_graph.model == m) & (~df_graph.matrix.isin(['B1', 'B2']))]
    if not len(sel):
        return None
    r = sel.sort_values('val_f1', ascending=False).iloc[0]
    return r['matrix'], r['alpha'], r['conf']

def _W(name, m):
    if name == 'D':
        return D_BY_MODEL.get(m), 'target'
    for n_, w_, l_ in MATS:
        if n_ == name:
            return w_, l_
    return None, None

# =============================================================================
# STEP 1 - TARGET-LEVEL RESULTS  (Sec 7.1)
# The 169-wide labels live as columns of df_test / df_val, not as an array.
# =============================================================================
print('=' * 78); print('STEP 1: target-level macro-F1'); print('=' * 78)

def _tgt_cols(df):
    """Find 169 columns that look like target labels (binary, target-id names)."""
    if df is None:
        return None
    cols = list(df.columns)
    # a) columns named exactly like the target ids
    byname = [c for c in cols if str(c) in set(TARGET_IDS)]
    if len(byname) == 169:
        return byname
    # b) columns named tgt_1.1 / target_1.1 / sdg_1.1 ...
    for pref in ['tgt_', 'target_', 'sdg_', 't_']:
        p = [c for c in cols if str(c).startswith(pref) and str(c)[len(pref):] in set(TARGET_IDS)]
        if len(p) == 169:
            return p
    # c) any contiguous run of 169 binary columns
    binc = []
    for c in cols:
        try:
            v = pd.to_numeric(df[c], errors='coerce').dropna().unique()
        except Exception:
            continue
        if len(v) and set(np.asarray(v).ravel().tolist()) <= {0, 1, 0.0, 1.0}:
            binc.append(c)
    if len(binc) >= 169:
        print(f'    {len(binc)} binary columns found; using the last 169')
        return binc[-169:]
    return None

_dte = globals().get('df_test'); _dva = globals().get('df_val')
ct, cv = _tgt_cols(_dte), _tgt_cols(_dva)

if ct is None or cv is None:
    print('  !! Could not identify 169 target-label columns.')
    print('     Paste the output of:  list(df_test.columns)')
    df_tgt = None
else:
    print(f'  using columns {ct[0]} ... {ct[-1]}')
    yt_te = _dte[ct].to_numpy().astype(int)
    yt_va = _dva[cv].to_numpy().astype(int)
    print(f'  target labels: test{yt_te.shape} val{yt_va.shape}, '
          f'positives/paper test={yt_te.sum(1).mean():.2f}')
    rows = []
    for m in MODELS:
        tv = cached[m]['target_val'].astype(np.float64)
        tt = cached[m]['target_test'].astype(np.float64)
        v, t, _ = fas(tv, yt_va, tt, yt_te)
        rows.append({'model': m, 'matrix': 'unpropagated', 'val_f1_tgt': v, 'test_f1_tgt': t})
        print(f'  {m:15s} unpropagated: {t:.4f}')
        for name in ['B', 'C', 'D', 'E', 'F', 'G']:
            sel = df_graph[(df_graph.model == m) & (df_graph.matrix == name)]
            if not len(sel):
                continue
            W, lvl = (D_BY_MODEL.get(m), 'target') if name == 'D' else \
                     next(((w, l) for n_, w, l in MATS if n_ == name), (None, None))
            if W is None or lvl != 'target':
                continue
            a, c = sel.iloc[0]['alpha'], sel.iloc[0]['conf']
            v, t, _ = fas(propagate(tv, W, a, c), yt_va, propagate(tt, W, a, c), yt_te)
            rows.append({'model': m, 'matrix': name, 'val_f1_tgt': v, 'test_f1_tgt': t})
            print(f'  {m:15s} {name}: {t:.4f}  (alpha={a}, conf={c})')
    df_tgt = pd.DataFrame(rows)
    df_tgt.to_csv(os.path.join(OUT, 'target_level.csv'), index=False)
    p = df_tgt[df_tgt.model == PRIMARY]
    if len(p):
        unp = p[p.matrix == 'unpropagated']['test_f1_tgt'].iloc[0]
        mats = p[p.matrix != 'unpropagated']
        print(f'\n  -> {PRIMARY}: unpropagated {unp:.4f}; matrices '
              f'{mats["test_f1_tgt"].min():.4f}-{mats["test_f1_tgt"].max():.4f} '
              f'(span {mats["test_f1_tgt"].max()-mats["test_f1_tgt"].min():.4f}); '
              f'best {mats.sort_values("test_f1_tgt").iloc[-1]["matrix"]}')

# =============================================================================
# STEP 2 - PER-GOAL BREAKDOWNS  (Sec 7.1 and 7.2)
# =============================================================================
print('\n' + '=' * 78); print('STEP 2: per-goal breakdowns'); print('=' * 78)

def per_goal(p_val, p_test):
    thr = fit_thresholds_fast(p_val, y_val)
    pred = (p_test >= thr).astype(int)
    _y = y_test.astype(bool); _p = pred.astype(bool)
    return np.array([_f1_bin(_y[:, c], _p[:, c]) for c in range(17)])

pg_rows = []
for m in MODELS:
    gv, gt = cached[m]['goal_val'].astype(np.float64), cached[m]['goal_test'].astype(np.float64)
    av = aggregate_targets_to_goals(cached[m]['target_val'])
    at = aggregate_targets_to_goals(cached[m]['target_test'])
    f_b1, f_b2 = per_goal(gv, gt), per_goal(av, at)

    vb = _valbest(m)
    f_gr = None
    if vb:
        name, a, c = vb
        W, lvl = _W(name, m)
        if W is not None:
            if lvl == 'goal':
                f_gr = per_goal(propagate(gv, W, a, c), propagate(gt, W, a, c))
            else:
                f_gr = per_goal(
                    aggregate_targets_to_goals(propagate(cached[m]['target_val'].astype(np.float64), W, a, c)),
                    aggregate_targets_to_goals(propagate(cached[m]['target_test'].astype(np.float64), W, a, c)))

    ev = nbr_vote(val_emb, pool_emb, pool_labels_goal, K_HEAD, True)
    et = nbr_vote(test_emb, pool_emb, pool_labels_goal, K_HEAD, True)
    best = None
    for lam in np.arange(0.0, 1.01, 0.1):
        v, t, _ = fas((1-lam)*gv + lam*ev, y_val, (1-lam)*gt + lam*et, y_test)
        if best is None or v > best[0]:
            best = (v, t, lam)
    lam = best[2]
    f_nb = per_goal((1-lam)*gv + lam*ev, (1-lam)*gt + lam*et)

    for c in range(17):
        pg_rows.append({'model': m, 'goal': GOAL_NAMES[c],
                        'B1': f_b1[c], 'B2': f_b2[c],
                        'graph': (f_gr[c] if f_gr is not None else np.nan),
                        'graph_matrix': (vb[0] if vb else None),
                        'nbr': f_nb[c],
                        'nbr_minus_B2': f_nb[c] - f_b2[c],
                        'graph_minus_B1': (f_gr[c] - f_b1[c]) if f_gr is not None else np.nan})
    if f_gr is not None:
        d = f_gr - f_b1
        print(f'  {m:15s} graph {vb[0]} vs B1: best +{d.max():.3f} ({GOAL_NAMES[d.argmax()]}), '
              f'worst {d.min():.3f} ({GOAL_NAMES[d.argmin()]})')
    d = f_nb - f_b2
    print(f'  {m:15s} nbr vs B2: {int((d>0).sum())}/17 goals improve; '
          f'max +{100*d.max():.1f}pp ({GOAL_NAMES[d.argmax()]}), '
          f'min {100*d.min():+.1f}pp ({GOAL_NAMES[d.argmin()]})')
df_pg = pd.DataFrame(pg_rows)
df_pg.to_csv(os.path.join(OUT, 'per_goal.csv'), index=False)

# =============================================================================
# STEP 3 - K x LAMBDA ABLATION  (Sec 7.4)
# =============================================================================
print('\n' + '=' * 78); print('STEP 3: K x lambda ablation'); print('=' * 78)

def per_goal_lambda_cv(prior_v, ev_v, yv, folds=5, seed=42):
    """Per-goal lambda by 5-fold CV; matches the released code's grid (0..1.0)."""
    from sklearn.model_selection import KFold
    n, C = prior_v.shape
    lams = np.zeros(C)
    grid = np.arange(0.0, 1.01, 0.05)
    for c in range(C):
        kf = KFold(n_splits=folds, shuffle=True, random_state=seed)
        picks = []
        for tr, te in kf.split(np.arange(n)):
            bl, bf = 0.0, -1.0
            for lam in grid:
                combo = (1-lam)*prior_v[te, c] + lam*ev_v[te, c]
                _y = yv[te, c].astype(bool)
                f = max(_f1_bin(_y, combo >= t) for t in CANON_GRID)
                if f > bf:
                    bf, bl = f, lam
            picks.append(bl)
        lams[c] = float(np.median(picks))
    return lams

abl = []
for m in MODELS:
    gv, gt = cached[m]['goal_val'].astype(np.float64), cached[m]['goal_test'].astype(np.float64)
    for K in [5, 10, 20, 50]:
        ev = nbr_vote(val_emb, pool_emb, pool_labels_goal, K, True)
        et = nbr_vote(test_emb, pool_emb, pool_labels_goal, K, True)
        # global
        best = None
        for lam in np.arange(0.0, 1.01, 0.1):
            v, t, _ = fas((1-lam)*gv + lam*ev, y_val, (1-lam)*gt + lam*et, y_test)
            if best is None or v > best[0]:
                best = (v, t, lam)
        # per-goal
        lg = per_goal_lambda_cv(gv, ev, y_val)
        cv_ = gv*(1-lg) + ev*lg
        ct_ = gt*(1-lg) + et*lg
        pv, pt, _ = fit_and_score(cv_, y_val, ct_, y_test)
        abl.append({'model': m, 'K': K,
                    'global_val': best[0], 'global_test': best[1], 'global_lambda': best[2],
                    'pergoal_val': pv, 'pergoal_test': pt,
                    'global_minus_pergoal_test': best[1] - pt})
        print(f'  {m:15s} K={K:<3d} global {best[1]:.4f} (lam={best[2]:.1f}) | '
              f'per-goal {pt:.4f} | diff {best[1]-pt:+.4f}')
df_abl = pd.DataFrame(abl)
df_abl.to_csv(os.path.join(OUT, 'ablation_K_lambda.csv'), index=False)
w = df_abl['global_minus_pergoal_test']
print(f'\n  -> global beats per-goal in {int((w>0).sum())}/{len(w)} comparisons; '
      f'mean {w.mean():+.4f}, min {w.min():+.4f}')

# =============================================================================
# STEP 4 - ENTITY EVIDENCE, doc_entities AND THREE-WAY  (Sec 7.2, Table 5)
# Uses the notebook's own entity evidence, not a rebuild from descriptors.
# =============================================================================
print('\n' + '=' * 78); print('STEP 4: entity evidence and fusion'); print('=' * 78)

Ev_va = globals().get('ent_evid_val')
Ev_te = globals().get('ent_evid_test')
if Ev_va is None or Ev_te is None:
    print('  !! ent_evid_val / ent_evid_test not in the namespace.')
    print('     Run the notebook cell that builds entity evidence, then re-run.')
else:
    Ev_va = np.asarray(Ev_va, dtype=np.float64)
    Ev_te = np.asarray(Ev_te, dtype=np.float64)
    print(f'  using ent_evid_val{Ev_va.shape}, ent_evid_test{Ev_te.shape}')
    nz = Ev_te > 0
    print(f'  coverage: {100*nz.mean():.1f}% of {Ev_te.size:,} paper-goal pairs are non-zero')
    gp = cached[PRIMARY]['goal_test'].astype(np.float64)
    cors = []
    for c in range(17):
        msk = nz[:, c]
        if msk.sum() > 10 and Ev_te[msk, c].std() > 0 and gp[msk, c].std() > 0:
            cors.append(float(np.corrcoef(Ev_te[msk, c], gp[msk, c])[0, 1]))
    if cors:
        print(f'  per-goal corr with LLM score: {min(cors):+.3f} to {max(cors):+.3f}, '
              f'{sum(1 for x in cors if x > 0)}/{len(cors)} positive')

    rows = []
    for m in MODELS:
        gv = cached[m]['goal_val'].astype(np.float64)
        gt = cached[m]['goal_test'].astype(np.float64)
        nv = nbr_vote(val_emb, pool_emb, pool_labels_goal, K_HEAD, True)
        nt = nbr_vote(test_emb, pool_emb, pool_labels_goal, K_HEAD, True)
        be = None
        for lam in np.arange(0.0, 1.01, 0.1):
            v, t, _ = fas((1-lam)*gv + lam*Ev_va, y_val, (1-lam)*gt + lam*Ev_te, y_test)
            if be is None or v > be[0]:
                be = (v, t, round(float(lam), 2))
        bt = None
        for ln in np.arange(0.0, 1.01, 0.1):
            for le in np.arange(0.0, 1.01 - ln + 1e-9, 0.1):
                lp = 1 - ln - le
                v, t, _ = fas(lp*gv + ln*nv + le*Ev_va, y_val,
                              lp*gt + ln*nt + le*Ev_te, y_test)
                if bt is None or v > bt[0]:
                    bt = (v, t, round(float(lp), 2), round(float(ln), 2), round(float(le), 2))
        rows.append({'model': m, 'entities_val': be[0], 'entities_test': be[1],
                     'lambda_ent_only': be[2], 'threeway_val': bt[0],
                     'threeway_test': bt[1], 'lambda_prior': bt[2],
                     'lambda_nbr': bt[3], 'lambda_ent': bt[4]})
        print(f'  {m:15s} entities {be[1]:.4f} | three-way {bt[1]:.4f} '
              f'(lp,ln,le)=({bt[2]},{bt[3]},{bt[4]})')
    df_fuse = pd.DataFrame(rows)
    df_fuse.to_csv(os.path.join(OUT, 'entities_and_fusion.csv'), index=False)


# =============================================================================
# STEP 5 - DIAGNOSTICS 1-3 AND THE PREVALENCE REGRESSION  (Sec 7.5)
# =============================================================================
print('\n' + '=' * 78); print('STEP 5: diagnostics'); print('=' * 78)

def _tri_corr(A, B):
    iu = np.triu_indices(A.shape[0], 1)
    a, b = A[iu], B[iu]
    ok = np.isfinite(a) & np.isfinite(b)
    if a[ok].std() == 0 or b[ok].std() == 0:
        return np.nan
    return float(np.corrcoef(a[ok], b[ok])[0, 1])

diag = {}
tp = cached[PRIMARY]['target_test'].astype(np.float64)
R = np.corrcoef(tp.T); np.fill_diagonal(R, 0.0); R = np.nan_to_num(R)
for nm, W in [('C', W_C), ('B', W_B)]:
    if W is not None:
        diag[f'r(R,W_{nm})'] = _tri_corr(R, W)
        print(f'  Diagnostic 1: r(R, W_{nm}) = {diag[f"r(R,W_{nm})"]:+.3f}')

def _top(M, k=50):
    iu = np.triu_indices(M.shape[0], 1)
    v = np.abs(M[iu]); idx = np.argsort(-v)[:k]
    return set(zip(iu[0][idx], iu[1][idx]))
tR = _top(R)
for nm, W in [('C', W_C), ('B', W_B)]:
    if W is not None:
        ov = len(tR & _top(W)) / 50
        diag[f'top50_overlap_{nm}'] = ov
        print(f'  Diagnostic 1: top-50 overlap with {nm} = {100*ov:.0f}%')

if Ev_te is not None:
    gp = cached[PRIMARY]['goal_test'].astype(np.float64)
    nz = Ev_te > 0
    diag['entity_coverage'] = float(nz.mean())
    print(f'  Diagnostic 2: entity coverage = {100*nz.mean():.1f}% of paper-goal pairs')
    cors = []
    for c in range(17):
        msk = nz[:, c]
        if msk.sum() > 10 and Ev_te[msk, c].std() > 0 and gp[msk, c].std() > 0:
            cors.append(float(np.corrcoef(Ev_te[msk, c], gp[msk, c])[0, 1]))
    if cors:
        diag['entity_corr_min'], diag['entity_corr_max'] = min(cors), max(cors)
        print(f'  Diagnostic 2: per-goal corr with LLM score '
              f'{min(cors):+.3f} to {max(cors):+.3f}, '
              f'{int(sum(1 for x in cors if x>0))}/{len(cors)} positive')

gv = cached[PRIMARY]['goal_val'].astype(np.float64)
gt = cached[PRIMARY]['goal_test'].astype(np.float64)
av = aggregate_targets_to_goals(cached[PRIMARY]['target_val'])
at = aggregate_targets_to_goals(cached[PRIMARY]['target_test'])
nv = nbr_vote(val_emb, pool_emb, pool_labels_goal, K_HEAD, True)
nt = nbr_vote(test_emb, pool_emb, pool_labels_goal, K_HEAD, True)
bl = None
for lam in np.arange(0.0, 1.01, 0.1):
    v, t, _ = fas((1-lam)*gv + lam*nv, y_val, (1-lam)*gt + lam*nt, y_test)
    if bl is None or v > bl[0]:
        bl = (v, t, lam)
lam = bl[2]
thr_n = fit_thresholds_fast((1-lam)*gv + lam*nv, y_val)
thr_b2 = fit_thresholds_fast(av, y_val)
pred_n = (((1-lam)*gt + lam*nt) >= thr_n).astype(int)
pred_b2 = (at >= thr_b2).astype(int)
ok_n, ok_b2 = (pred_n == y_test), (pred_b2 == y_test)
WR = (~ok_b2) & ok_n
RW = ok_b2 & (~ok_n)
diag['WR_total'], diag['RW_total'] = int(WR.sum()), int(RW.sum())
print(f'  Diagnostic 3: fixes {WR.sum()}, breaks {RW.sum()}, net {WR.sum()-RW.sum():+d}')

p = np.clip(gt, 1e-9, 1-1e-9)
ent = -(p*np.log2(p) + (1-p)*np.log2(1-p)).mean(axis=1)   # mean binary entropy
sims = test_emb @ pool_emb.T
topk = np.partition(sims, -K_HEAD, axis=1)[:, -K_HEAD:]
rel = topk.mean(axis=1)
hu, hr = ent > np.median(ent), rel > np.median(rel)
flip = []
for lab, msk in [('uncertain x reliable', hu & hr), ('uncertain x unreliable', hu & ~hr),
                 ('certain x reliable', ~hu & hr), ('certain x unreliable', ~hu & ~hr)]:
    flip.append({'stratum': lab, 'n': int(msk.sum()),
                 'WR': float(WR[msk].sum()/max(msk.sum(), 1)),
                 'RW': float(RW[msk].sum()/max(msk.sum(), 1))})
    flip[-1]['net'] = flip[-1]['WR'] - flip[-1]['RW']
    print(f'    {lab:24s} n={flip[-1]["n"]:5d} WR={flip[-1]["WR"]:.3f} '
          f'RW={flip[-1]["RW"]:.3f} net={flip[-1]["net"]:+.3f}')
pd.DataFrame(flip).to_csv(os.path.join(OUT, 'flip_buckets.csv'), index=False)

prev = y_test.mean(axis=0)
lift = df_pg[df_pg.model == PRIMARY]['nbr_minus_B2'].values
if prev.std() > 0 and lift.std() > 0:
    diag['prevalence_r'] = float(np.corrcoef(prev, lift)[0, 1])
    print(f'  Prevalence vs lift: r = {diag["prevalence_r"]:+.3f} on {PRIMARY}')

# =============================================================================
# STEP 6 - AURORA CO-OCCURRENCE GRAPH  (Sec 7.5, Sec 8.2)
# =============================================================================
print('\n' + '=' * 78); print('STEP 6: Aurora co-occurrence graph'); print('=' * 78)
L = np.asarray(pool_labels_goal, dtype=np.float64)
P_i = L.mean(axis=0)
P_ij = (L.T @ L) / L.shape[0]
P_either = P_i[:, None] + P_i[None, :] - P_ij
W_aur = np.where(P_either > 0, P_ij/P_either, 0) - P_i[:, None]*P_i[None, :]
np.fill_diagonal(W_aur, 0.0)
print('  built from the neighbour pool only (disjoint from few-shot/val/test)')

aur = []
for m in MODELS:
    gv_, gt_ = cached[m]['goal_val'].astype(np.float64), cached[m]['goal_test'].astype(np.float64)
    _, b1, _ = fas(gv_, y_val, gt_, y_test)
    best = None
    for a, c in itertools.product(ALPHAS, CONFS):
        v, t, _ = fas(propagate(gv_, W_aur, a, c), y_val,
                                propagate(gt_, W_aur, a, c), y_test)
        if best is None or v > best[0]:
            best = (v, t, a, c)
    aur.append({'model': m, 'B1': b1, 'aurora_test': best[1],
                'delta_vs_B1': best[1]-b1, 'alpha': best[2], 'conf': best[3]})
    print(f'  {m:15s} aurora {best[1]:.4f}  vs B1 {best[1]-b1:+.4f}')
pd.DataFrame(aur).to_csv(os.path.join(OUT, 'aurora_graph.csv'), index=False)

with open(os.path.join(OUT, 'diagnostics.json'), 'w') as f:
    json.dump(diag, f, indent=2)

print('\n' + '=' * 78)
print('DONE. Everything in the paper is now produced by one routine.')
for f in ['target_level.csv', 'per_goal.csv', 'ablation_K_lambda.csv',
          'entities_and_fusion.csv', 'flip_buckets.csv', 'aurora_graph.csv',
          'diagnostics.json']:
    print('  ', f)
print('=' * 78)